# Author profile: papers, patents and collaboration networks of one OpenAlex author

**Input:** one OpenAlex `author_id` (e.g. `A5067184382`, Jennifer A. Doudna). For Nobel laureates,
`output/nobel_laureates/laureate_author_map.csv` (from `nobel_laureate_papers.ipynb`) gives the id and
any fragment ids of the same person.
**Output:** every work the person authored (OpenAlex) and every patent they invented (PatentsView),
with the metrics of the *Science of Science* pipelines, a career summary, the patent↔paper links, the
PPP pairs, and the co-authorship and co-invention ego networks — tables, figures and a manifest in
`Nobel Prize/output/<author_id>/`.

```mermaid
flowchart LR
    A["author_id"] -->|pqrs_dataset.tsv<br/>(PatentsView 2023-03-30 ids)| I0["inventor ids 2023"]
    I0 -->|g_persistent_inventor| I["inventor ids 2025-12-31"]
    A -.->|no pqrs id: name search + evidence (2b)| I
    A -->|authorships_parquet| P["works"]
    I -->|g_inventor_disambiguated| T["patents"]
    P --> PM["paper metrics (np_common)"]
    T --> TM["patent metrics"]
    PM & TM --> S["summary + figures"]
    PM & TM --> X["PCS citations, PPP pairs"]
    P --> CA["co-authorship network"]
    T --> CI["co-invention network"]
    CA & CI --> O["people on both sides<br/>(pqrs + names)"]
```

## Data sources

| Block | File | What is taken |
|---|---|---|
| Crosswalk | `Scientist_Inventor/Data/pqrs_dataset.tsv` | candidate (author, inventor) pairs with PQRS `confidence` (share of matched comparisons), `number of match`, names. Many-to-many, inventor ids of the PatentsView **2023-03-30** disambiguation, translated to the current (2025-12-31) ids through `g_persistent_inventor`. |
| Name search | `PatentView/Granted/g_inventor_disambiguated.tsv.zip` (+ `g_assignee_disambiguated`, `g_location_disambiguated`) | inventors named like the author when pqrs has none, with co-inventor / citation / assignee / country evidence (section 2b) |
| Works | `Scientist_Inventor/Data/authorships_parquet/` | OpenAlex `works_au_affs` (1.0 B rows, integer ids, bucketed by `work_id % 64`); scanned for the author's works and for all authors of those works |
| Paper metrics | `Science of Science/OpenAlex/output/`, `pcs/output/`, `sciscinet_papers_fos_feg.parquet` | through `np_common.assemble_paper_metrics` / `finish_paper_metrics` — the same joins as `nobel_laureate_papers.ipynb` |
| Paper DA | `Atypicality/Data/_traj_cache/paper_year_DA/all_years/paper_metrics.parquet` | census OA_BERT PPPL (2000 / 2015 / 2020), turned into a shared population table (`cache/paper_da_population.v1.parquet`) |
| Patents | `PatentView/Granted/g_inventor_disambiguated`, `g_patent`, `g_persistent_inventor`; `Pregranted/pg_granted_pgpubs_crosswalk`, `pg_inventor_disambiguated` | inventor → patents, names, titles; release translation; pre-grant publications |
| Patent metrics | `Science of Science/PatentView/output/`, `Atypicality/Data/rev/patent_master.parquet` | citations, percentiles, CD/F/E/G, Kim z, SB, countries, DA (shared table `cache/patent_da_population.v1.parquet`) |
| Links | `pcs/pcs_oa_uspto.csv` (Reliance on Science), `Atypicality/Data/ppp_master_rev.parquet` | patent→paper citations of the works and of the patents; PPP pairs |
| Trajectories | `paper_citation_trend`, `pcs_citation_trend`, `patent_citation_trend`, `patent_uniqueC_trend` | yearly citations received |

## Measures — every one as a cohort percentile

Each document is compared with the documents of its own cohort: **works** with the papers of the same
publication year × first OpenAlex FoS field, **patents** with the utility patents of the same grant year ×
CPC Section (patent DA: filing year × CPC Section). A percentile is the mid-rank share of the cohort below the document (0.5 = a typical
document of the cohort, ties split evenly); "top 10 %" / "top 1 %" count documents whose percentile is
≥ 0.90 / 0.99 (a large tie block such as the never-cited majority sits at its mid-point, never at the top;
the ties-inclusive `_cume` columns are kept in the tables). The shared population tables live in
`Nobel Prize/cache/`.

| Measure (key) | Works: column (raw value) | Patents: column (raw value) |
|---|---|---|
| Impact | `C_5_pctl` (5-year citations `C_5`), `C_10_pctl` | `uniqueC_5_pctl` (5-year citations, granted + pre-grant, `uniqueC_5`; cohort = patents whose window closed) |
| Science ↔ technology | `pcs_C_5_pctl` (patents citing the work within 5 years, 0 for the uncited majority) | `n_paper_refs_pctl` (papers the patent cites, Reliance on Science) |
| Disruption | `CD_5_rpctl_mid` (CD_5, cohort = papers **with references**: a reference-less document has CD = 1 by construction and is blanked) | `CD_5_rpctl_mid` (CD_5, patents with references and a closed window) |
| Foundation / Extension / Generalization | `F_5_pctl`, `E_5_pctl`, `G_5_pctl` (citer shares) | same |
| Novelty | `novelty_pctl` = 1 − percentile of Uzzi `Z_10pct` (high = atypical journal combinations) | same with Kim's CPC-pair z |
| Conventionality | `Z_median_pctl` (Uzzi `Z_median`) | same (Kim) |
| Discursive atypicality | `DA_pctl` (OA_BERT PPPL within scoring year × first FoS, census article population 2000 / 2015 / 2020 only) | `DA_pctl` (within filing year × CPC Section, all scored patents) |
| Sleeping beauty | `SB_B_pctl` | `SB_B_pctl` |
| Textbook reach | `book_cites_pctl` (distinct books, book chapters and reference entries citing the work; 0 for the never-book-cited majority) | — |

Only documents up to `YEAR_MAX` (2021) are used: works by publication year, patents by grant year
(`PATENT_YEAR_BASIS = 'filing'` switches to the filing year). Window measures are kept up to
`WINDOW_END_YEAR` (2026, the snapshot year): the 5-year window of a 2021 document ends in the snapshot, and
because every percentile compares a document with its own cohort, all documents of a cohort share the same
window. The 10-year work percentile covers works up to 2016. Raw values stay in the tables. Author-position
roles (first / last author, first-named inventor) are not analysed.

Section 6b measures **textbook reach**: how often books, book chapters and reference entries cite each work.
Section 9b draws the collaboration networks with countries, institutions and assignees as nodes, and
section 10 assembles everything into one standalone interactive dashboard in `output/dashboard/`.

Every figure is saved twice: `figures/*.png` and an interactive `html/*.html` (plotly: hover for the
document or person, zoom, pan, legend toggles; `html/plotly.min.js` must stay next to the pages).

## How to run

Kernel `Python (Curvature)`. Run on a compute node (8 cores / 96 GB): the authorship scans read a
1.0 B-row dataset and the metric joins files of 2–9 GB. A new author takes 3–5 minutes; the first run
ever also builds the shared caches in `Nobel Prize/cache/` (~5 minutes, mostly the inventor release map
and the pre-grant tables). Every heavy result is cached per author in `output/<author_id>/_cache/`,
keyed on the inputs (ids, inventor selection) and invalidated when a source file changes.
`sbatch --export=ALL,NP_AUTHOR_ID=A... jobs/author_profile.sbatch` runs a copy of the notebook.

## 0. Parameters

In [ ]:
import os

AUTHOR_ID = os.environ.get('NP_AUTHOR_ID', 'A5067184382').strip()     # Jennifer A. Doudna (Nobel Chemistry 2020)
EXTRA_AUTHOR_IDS = [a.strip() for a in os.environ.get('NP_EXTRA_AUTHOR_IDS', '').split(';') if a.strip()]  # fragments of the same person
INVENTOR_IDS = None            # current-release (PatentsView 2025-12-31) inventor ids; bypasses the crosswalk
YEAR_MAX = int(os.environ.get('NP_YEAR_MAX', '2021'))   # only works published and patents granted up to this year
WINDOW_END_YEAR = 2026         # window measures are kept when year + W <= this (the snapshot year; percentiles compare within cohort)
BOOK_MIN = 3                   # 'textbook reach': works cited by at least this many books / chapters / reference entries
DASH_MEASURES = ['impact5', 'impact10', 'F', 'E', 'G', 'disruption', 'books']   # measures on the dashboard (all stay in the tables, PNGs, html/)
PATENT_YEAR_BASIS = 'grant'    # 'grant' or 'filing': the patent year compared with YEAR_MAX
INVENTOR_SELECT = 'threshold'  # 'threshold' (all candidates with confidence >= MIN_CONFIDENCE), 'best', 'all', 'greedy'
MIN_CONFIDENCE = 0.5
MIN_RELEASE_SHARE = 0.2        # a current inventor id is kept when it carries >= this share of a pqrs id's patent rows
NAME_SEARCH = os.environ.get('NP_NAME_SEARCH', 'auto')   # PatentsView name search: 'auto' (only when pqrs gives no inventor), 'always', 'off'
ID_CHECK = os.environ.get('NP_ID_CHECK', 'auto')         # author id check (section 0b): 'auto' or 'off'
MIN_WORKS = 5                  # an id with fewer works in the 2026-01 snapshot is resolved (ORCID, laureate bridge, name)
TITLES_TABLE = os.environ.get('NP_TITLES_TABLE', '').strip()   # titles: a shared (id, title) parquet; 'api' = OpenAlex API only; empty = scan the raw works table
NOBEL_LOOKUP = os.environ.get('NP_NOBEL_LOOKUP', 'auto')       # Nobel prize lookup: 'auto' (ids, ORCID, then name), 'ids' (no name match), 'off'
QUERY = os.environ.get('NP_QUERY', '').strip()                 # the name typed into pipeline/profile_person.py, if any
RESOLUTION = os.environ.get('NP_RESOLUTION', '').strip()       # how the pipeline chose the author id

RUN_TRAJECTORY = True
FETCH_TITLES = os.environ.get('NP_FETCH_TITLES', '0') == '1'   # titles of the works and citing books (205 GB scan of the raw works table, ~5 min)
TOP_K_COLLAB = 40              # collaborators drawn per network figure
LABEL_TOP = 12                 # labelled nodes per network figure
HTML_TOP_K = 150               # collaborators drawn in the interactive network pages
MAX_TEAM_SIZE = 50             # documents with more people are left out of the network graphs (consortia)
OBS_END_YEAR = 2025            # last complete year of the 2026-01 snapshots
COVERAGE_END = 2022            # last citing year with complete reference coverage in the OpenAlex graph
CD_REQUIRE_REFS = True         # blank the disruption family of documents without references
RESEARCH_ONLY = True           # headline statistics over articles, reviews and letters
PRIZE_YEAR = os.environ.get('NP_PRIZE_YEAR', '').strip()     # dashboard file name <year>_<field>_<name>_<id>.html; empty = looked up
PRIZE_FIELD = os.environ.get('NP_PRIZE_FIELD', '').strip()   # (PrizeAtlas crawl in Data/prizeatlas, then the laureate author bridge)
REBUILD_CACHE = False
print(dict(AUTHOR_ID=AUTHOR_ID, EXTRA_AUTHOR_IDS=EXTRA_AUTHOR_IDS, YEAR_MAX=YEAR_MAX, WINDOW_END_YEAR=WINDOW_END_YEAR, BOOK_MIN=BOOK_MIN,
           PATENT_YEAR_BASIS=PATENT_YEAR_BASIS, INVENTOR_IDS=INVENTOR_IDS, INVENTOR_SELECT=INVENTOR_SELECT,
           MIN_CONFIDENCE=MIN_CONFIDENCE, MIN_RELEASE_SHARE=MIN_RELEASE_SHARE, NAME_SEARCH=NAME_SEARCH, RUN_TRAJECTORY=RUN_TRAJECTORY,
           FETCH_TITLES=FETCH_TITLES, TOP_K_COLLAB=TOP_K_COLLAB, MAX_TEAM_SIZE=MAX_TEAM_SIZE, OBS_END_YEAR=OBS_END_YEAR, COVERAGE_END=COVERAGE_END,
           CD_REQUIRE_REFS=CD_REQUIRE_REFS, RESEARCH_ONLY=RESEARCH_ONLY, REBUILD_CACHE=REBUILD_CACHE))
if 'SLURM_JOB_ID' not in os.environ:
    print('WARNING: not inside a Slurm allocation. The login node caps the user at 8 GiB; run this notebook on a compute node.')

### Setup

In [ ]:
import sys, re, json, time, math, datetime, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import ListedColormap, to_hex
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

sys.path.insert(0, '/project/jevans/Dawoon/Nobel Prize/notebook')
import np_common as npc

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 200, 'display.max_columns', 60, 'display.max_colwidth', 60)
npc.style()
THREADS, DUCKDB_MEM = npc.resources()
con = npc.duck(THREADS, DUCKDB_MEM)

AUTHOR_IDS = list(dict.fromkeys([AUTHOR_ID] + EXTRA_AUTHOR_IDS))
for a in AUTHOR_IDS:
    assert re.fullmatch(r'A\d+', a), f'not an OpenAlex author id: {a}'
AUTHOR_INTS = sorted(int(a[1:]) for a in AUTHOR_IDS)

OUT = npc.NP / 'output' / AUTHOR_ID
FIG = OUT / 'figures'
HTML = OUT / 'html'
for d in (OUT, FIG, HTML):
    d.mkdir(parents=True, exist_ok=True)
CACHE_VERSION = 5      # bump when a cached builder changes (the key covers inputs, not code)
cache = npc.Cache(OUT / '_cache', version=CACHE_VERSION, rebuild=REBUILD_CACHE)
WRITTEN, TIMINGS = [], {}

def savefig(fig, name):
    p = FIG / f'{name}.png'
    fig.savefig(p, bbox_inches='tight'); WRITTEN.append(p)
    print('figure ->', p.relative_to(npc.NP))

def save_html(fig, name):
    # interactive page; plotly.min.js is written once next to the pages (include_plotlyjs='directory'), so they open offline
    fig.update_layout(template='plotly_white', font=dict(family='Arial, Helvetica, sans-serif', size=12, color=npc.INK),
                      margin=dict(l=70, r=40, t=90, b=60))
    p = HTML / f'{name}.html'
    fig.write_html(p, include_plotlyjs='directory', config={'displaylogo': False, 'scrollZoom': True,
                                                           'toImageButtonOptions': {'format': 'png', 'scale': 2}})
    WRITTEN.append(p); DASH[name] = fig; print('html   ->', p.relative_to(npc.NP))

DASH = {}          # every interactive figure, reused by the dashboard (section 10)

def dash_only(fig, name):
    # a dashboard-only variant of a figure (same styling as save_html; the html/ page keeps the full figure)
    fig.update_layout(template='plotly_white', font=dict(family='Arial, Helvetica, sans-serif', size=12, color=npc.INK),
                      margin=dict(l=70, r=40, t=90, b=60))
    DASH[name] = fig


def doc_hover(frame, side):
    # hover text of each document for the interactive pages
    if side == 'works':
        t = frame.title.astype('string').fillna('').str.slice(0, 80) + '<br>' if 'title' in frame else ''
        return (t + frame.paper_id + '<br>' + frame.year.astype('string').fillna('?') + ' · ' + frame.journal.astype('string').fillna('').str.slice(0, 60)
                + '<br>' + frame.doctype.astype('string').fillna(''))
    return (frame.patent_id + '<br>' + frame.basis_year.astype('Int64').astype('string').fillna('?') + ' · '
            + frame.patent_title.astype('string').fillna('').str.slice(0, 70))

def save_table(df, name, index=False):
    p = OUT / f'{name}.csv'
    df.to_csv(p, index=index); WRITTEN.append(p)
    print(f'table  -> {p.relative_to(npc.NP)}  ({len(df):,} rows)')

def save_parquet(df, name):
    p = OUT / f'{name}.parquet'
    df.to_parquet(p, index=False); WRITTEN.append(p)

PAPER_SOURCES = [npc.OA / f for f in ('paper_metadata.parquet', 'paper_citation.parquet', 'paper_hit_probability.parquet', 'paper_disruption.parquet',
                                      'paper_z_score.parquet', 'paper_sb.parquet', 'paper_author_country.parquet')] + \
                [npc.PCSDIR / 'output' / 'pcs_citation.parquet', npc.PCSDIR / 'output' / 'pcs_hit_probability.parquet', npc.PPP_MASTER, npc.SSN_PAPERS]
PATENT_SOURCES = [npc.PV / f for f in ('patent_metadata.parquet', 'patent_citation.parquet', 'patent_hit_probability.parquet', 'patent_disruption.parquet',
                                       'patent_z_score.parquet', 'patent_sb.parquet', 'patent_inventor_country.parquet')] + [npc.PPP_MASTER]
AUTH_SOURCES = [npc.AUTHORSHIPS / 'manifest.json']
SC, TC, LC, GREY = npc.SCI, npc.TECH, npc.LINK, npc.GREY
print('output ->', OUT)

## 0b. Author id check

An OpenAlex author id can carry no works in the 2026-01 snapshot that the whole notebook reads: OpenAlex creates, splits and
merges author ids after a snapshot (PrizeAtlas links the live ids), and an id can belong to a fragment of the person.
`np_common.resolve_author_ids` checks the ids against the snapshot's authors table (`OpenAlex_2026_Jan_16_Renly_parquet/authors.csv.gz`,
cached as `cache/openalex_authors.parquet`):

* ids with at least `MIN_WORKS` works are kept, and other ids carrying the same ORCID are added (OpenAlex splits people);
* otherwise the person is looked up by ORCID (the snapshot's and PrizeAtlas'), then through the Li et al. laureate bridge,
  then by name (same first + last name, prize-time institution, most cited, flagged when ambiguous).

The id with the most works becomes `AUTHOR_ID` (the output folder and the dashboard name); `NP_ID_CHECK=off` skips the check.

In [ ]:
INPUT_AUTHOR_IDS = list(AUTHOR_IDS)
ID_CHECK_RESULT = {'rule': 'not checked', 'input_ids': INPUT_AUTHOR_IDS, 'changed': False}
if ID_CHECK != 'off':
    hints = npc.laureate_hints(AUTHOR_IDS)             # PrizeAtlas: name, ORCID, prize-time institution, Li et al. bridge ids
    res = npc.resolve_author_ids(con, AUTHOR_IDS, min_works=MIN_WORKS, name=hints['name'], orcids=hints['orcids'],
                                 institution_ids=hints['institution_ids'], bridge_ids=hints['bridge_ids'])
    display(res['table'].head(20))
    ID_CHECK_RESULT = {k: v for k, v in res.items() if k != 'table'}
    if res['changed'] and res['author_ids']:
        old_out = OUT
        AUTHOR_IDS = res['author_ids']; AUTHOR_ID, EXTRA_AUTHOR_IDS = AUTHOR_IDS[0], AUTHOR_IDS[1:]
        AUTHOR_INTS = sorted(int(a[1:]) for a in AUTHOR_IDS)
        OUT = npc.NP / 'output' / AUTHOR_ID
        FIG, HTML = OUT / 'figures', OUT / 'html'
        for d in (OUT, FIG, HTML):
            d.mkdir(parents=True, exist_ok=True)
        cache = npc.Cache(OUT / '_cache', version=CACHE_VERSION, rebuild=REBUILD_CACHE)
        print(f'author ids: {INPUT_AUTHOR_IDS} -> {AUTHOR_IDS} [{res["rule"]}]; output -> {OUT.relative_to(npc.NP)}')
else:
    print('author id check off')

## 1. Crosswalk: OpenAlex author → PatentsView inventor(s)

`pqrs_dataset.tsv` is many-to-many. Candidates are ranked by PQRS `confidence` (the share of the
comparisons behind the pair that matched), then `number of match`, `name_similarity` and the id. The
default rule keeps **every candidate with confidence ≥ `MIN_CONFIDENCE`** — PatentsView splits some
people into several inventor ids, and a single "best" id can be a small fragment. The pqrs ids belong
to the PatentsView 2023-03-30 disambiguation; PatentsView relabels ids between releases, so each is
translated to the current (2025-12-31) ids that carry its patent rows (`g_persistent_inventor`), and
the current names are checked against the pqrs name. The reverse lookup lists other OpenAlex author
ids linked to the same inventors (candidates for `EXTRA_AUTHOR_IDS`). When the crosswalk gives no inventor id,
section 2b searches PatentsView directly by name and keeps only candidates with independent evidence.

In [ ]:
# shared lookups: pqrs as parquet, PatentsView inventor table and patent titles as parquet
PQRS_PQ = npc.CACHE / 'pqrs_dataset.parquet'
if not npc.fresh(PQRS_PQ, [npc.PQRS_TSV]):
    npc.copy_atomic(con, f'''SELECT inventor_id, author_id, "number of comparison" AS n_comparison, "number of match" AS n_match,
                                    scientist_name, inventor_name, name_similarity, flag,
                                    "number of papers" AS n_papers_pqrs, "number of patents" AS n_patents_pqrs, confidence
                             FROM read_csv('{npc.PQRS_TSV}', delim = '\t', header = true)''', PQRS_PQ)
con.execute(f"CREATE OR REPLACE VIEW pqrs AS SELECT * FROM read_parquet('{PQRS_PQ}')")

READ = dict(sep='\t', dtype=str, keep_default_na=False, na_values=[''])
def tsv_zip_to_parquet(src, dst, usecols, chunksize=2_000_000):
    # every column as a string; written through a temporary file
    t0 = time.time(); writer = None; n = 0
    schema = pa.schema([(c, pa.string()) for c in usecols])
    tmp = f'{dst}.{os.getpid()}.tmp'
    for ch in pd.read_csv(src, usecols=usecols, chunksize=chunksize, **READ):
        tbl = pa.Table.from_pandas(ch[usecols], schema=schema, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(tmp, schema, compression='zstd')
        writer.write_table(tbl); n += len(ch)
    writer.close(); os.replace(tmp, dst)
    print(f'{Path(dst).name}: {n:,} rows in {time.time() - t0:.0f}s')

LOOKUPS = {
    'g_inventor_disambiguated.parquet': (npc.granted('g_inventor_disambiguated.tsv.zip'),
        ['patent_id', 'inventor_sequence', 'inventor_id', 'disambig_inventor_name_first', 'disambig_inventor_name_last', 'gender_code', 'location_id']),
    'g_patent_min.parquet': (npc.granted('g_patent.tsv.zip'), ['patent_id', 'patent_type', 'patent_date', 'patent_title', 'num_claims', 'withdrawn']),
    'pg_inventor_disambiguated.parquet': (npc.pregranted('pg_inventor_disambiguated.tsv.zip'),
        ['pgpub_id', 'inventor_sequence', 'inventor_id', 'disambig_inventor_name_first', 'disambig_inventor_name_last']),
    'pg_granted_pgpubs_crosswalk.parquet': (npc.pregranted('pg_granted_pgpubs_crosswalk.tsv.zip'), ['pgpub_id', 'application_id', 'patent_id']),
    'g_assignee_disambiguated.parquet': (npc.granted('g_assignee_disambiguated.tsv.zip'),
        ['patent_id', 'assignee_sequence', 'assignee_id', 'disambig_assignee_individual_name_first', 'disambig_assignee_individual_name_last',
         'disambig_assignee_organization', 'assignee_type']),
}
for name, (src, cols) in LOOKUPS.items():
    if not npc.fresh(npc.CACHE / name, [src]):
        tsv_zip_to_parquet(src, npc.CACHE / name, cols)
con.execute(f'''CREATE OR REPLACE VIEW inv AS SELECT patent_id, TRY_CAST(inventor_sequence AS INTEGER) AS inventor_sequence, inventor_id,
                       disambig_inventor_name_first AS name_first, disambig_inventor_name_last AS name_last, gender_code
                FROM read_parquet('{npc.CACHE / 'g_inventor_disambiguated.parquet'}')''')
con.execute(f'''CREATE OR REPLACE VIEW gpat AS SELECT patent_id, regexp_replace(patent_id, '^H0+', 'H') AS patent_key, patent_type,
                       TRY_CAST(patent_date AS DATE) AS patent_date, patent_title, TRY_CAST(num_claims AS INTEGER) AS num_claims
                FROM read_parquet('{npc.CACHE / 'g_patent_min.parquet'}')''')
con.execute(f"CREATE OR REPLACE VIEW pginv AS SELECT pgpub_id, inventor_id FROM read_parquet('{npc.CACHE / 'pg_inventor_disambiguated.parquet'}')")
con.execute(f'''CREATE OR REPLACE VIEW pgx AS SELECT DISTINCT pgpub_id, patent_id FROM read_parquet('{npc.CACHE / 'pg_granted_pgpubs_crosswalk.parquet'}')
                WHERE pgpub_id IS NOT NULL AND patent_id IS NOT NULL''')

cand = con.sql(f'''SELECT * FROM pqrs WHERE author_id IN ({npc.sql_list(AUTHOR_IDS)})
                   ORDER BY confidence DESC, n_match DESC, name_similarity DESC, inventor_id''').df()
print(f'{len(cand)} candidate inventor rows for {AUTHOR_IDS}')
display(cand)

if INVENTOR_IDS:
    sel_pqrs, sel_rule = [], 'explicit INVENTOR_IDS (current release)'
elif INVENTOR_SELECT == 'greedy':
    g = pd.read_parquet(npc.GREEDY_MAP, columns=['author_id', 'inventor_id'])
    sel_pqrs, sel_rule = g.loc[g.author_id.isin(AUTHOR_IDS), 'inventor_id'].tolist(), 'greedy 1:1 mapping (Scientist_Inventor notebook 03)'
elif INVENTOR_SELECT == 'all':
    sel_pqrs, sel_rule = cand.inventor_id.drop_duplicates().tolist(), 'all candidates'
elif INVENTOR_SELECT == 'best':
    sel_pqrs, sel_rule = cand.inventor_id.head(1).tolist(), 'highest confidence'
else:
    sel_pqrs, sel_rule = cand.loc[cand.confidence >= MIN_CONFIDENCE, 'inventor_id'].drop_duplicates().tolist(), f'confidence >= {MIN_CONFIDENCE}'
sel_conf = cand.loc[cand.inventor_id.isin(sel_pqrs), 'confidence']
if len(cand) and not len(sel_pqrs) and not INVENTOR_IDS:
    print(f'WARNING: no candidate passes the rule ({sel_rule}); best candidate {cand.inventor_id.iloc[0]} has confidence {cand.confidence.iloc[0]:.3f}. '
          f'Set INVENTOR_SELECT="best" or INVENTOR_IDS to use it.')
if len(sel_pqrs) and len(cand.loc[~cand.inventor_id.isin(sel_pqrs)]) and \
        cand.loc[~cand.inventor_id.isin(sel_pqrs), 'n_match'].max() > cand.loc[cand.inventor_id.isin(sel_pqrs), 'n_match'].max():
    print('WARNING: an unselected candidate has more matched comparisons than every selected one — check the table above.')

# release translation 2023-03-30 -> current
rmap, REL_OLD, REL_NEW = npc.inventor_release_map()
if INVENTOR_IDS:
    inv_sel = list(dict.fromkeys(INVENTOR_IDS))
    translation = pd.DataFrame({'id_old': None, 'id_new': inv_sel})
else:
    translation = rmap[rmap.id_old.isin(sel_pqrs)].sort_values(['id_old', 'n_rows'], ascending=[True, False])
    display(translation)
    lost = sorted(set(sel_pqrs) - set(translation.id_old))
    if lost:
        print(f'WARNING: pqrs ids without any patent row in release {REL_OLD}: {lost}')
    inv_sel = translation.loc[translation.share_of_old >= MIN_RELEASE_SHARE, 'id_new'].drop_duplicates().tolist()
print(f'selected pqrs ids ({REL_OLD}) [{sel_rule}]: {sel_pqrs}\n-> current inventor ids ({REL_NEW}): {inv_sel}')

# name check of the current ids against the pqrs / OpenAlex names
cur_names = con.sql(f'''SELECT inventor_id, mode(trim(coalesce(name_first, '') || ' ' || coalesce(name_last, ''))) AS current_name, count(DISTINCT patent_id) AS n_patents
                        FROM inv WHERE inventor_id IN ({npc.sql_list(inv_sel)}) GROUP BY 1''').df()
ref_names = list(cand.scientist_name.dropna().unique()) + list(cand.inventor_name.dropna().unique())
cur_names['name_agrees'] = cur_names.current_name.map(lambda n: any(npc.names_agree(n, r) for r in ref_names) if ref_names else True)
display(cur_names)
if (~cur_names.name_agrees).any():
    print('WARNING: current inventor ids whose name does not match the author — dropped:', cur_names.loc[~cur_names.name_agrees, 'inventor_id'].tolist())
    inv_sel = cur_names.loc[cur_names.name_agrees, 'inventor_id'].tolist()
if not inv_sel:
    print('No inventor id: the patent side of this notebook will be empty.')

if npc.GREEDY_MAP.exists():
    g = pd.read_parquet(npc.GREEDY_MAP, columns=['author_id', 'inventor_id', 'confidence', 'number of match']).rename(columns={'number of match': 'n_match'})
    print('greedy 1:1 mapping rows for the author / the selected pqrs ids:')
    display(g[g.author_id.isin(AUTHOR_IDS) | g.inventor_id.isin(sel_pqrs)])
if sel_pqrs:
    rev = con.sql(f'''SELECT inventor_id, author_id, scientist_name, confidence, n_match, n_papers_pqrs FROM pqrs
                      WHERE inventor_id IN ({npc.sql_list(sel_pqrs)}) AND author_id NOT IN ({npc.sql_list(AUTHOR_IDS)}) ORDER BY confidence DESC''').df()
    print(f'{len(rev)} other author ids linked to the selected inventors (candidates for EXTRA_AUTHOR_IDS, not merged):')
    display(rev.head(20))
crosswalk = cand.assign(selected=cand.inventor_id.isin(sel_pqrs), rule=sel_rule)
save_table(crosswalk, 'crosswalk_candidates')
save_table(translation, 'inventor_id_translation')

## 2. The author's works

One scan of the authorship dataset (`author_id` filter) gives every (work, affiliation) row of the
person; a second scan (`work_id` filter) returns every author of those works, reused for the
co-authorship and institution networks. Author-position roles are not analysed.

In [ ]:
def scan_ego_authorships():
    return con.sql(f'''SELECT work_id, author_id, author_position_int, is_corresponding, institution_id, author_display_name, orcid
                       FROM read_parquet('{npc.AUTHORSHIPS}/bucket=*/*.parquet', hive_partitioning = true)
                       WHERE author_id IN ({', '.join(map(str, AUTHOR_INTS))})''').df()
t0 = time.time()
ego_rows = cache('ego_authorships', scan_ego_authorships, key=AUTHOR_INTS, sources=AUTH_SOURCES)
TIMINGS['authorship_scan_author'] = round(time.time() - t0, 1)
ego_rows['paper_id'] = 'W' + ego_rows.work_id.astype('int64').astype(str)
ego_papers = (ego_rows.groupby('paper_id', as_index=False)
              .agg(institution_ids=('institution_id', lambda s: ';'.join(sorted(set(x for x in s if x))))))
ego_papers['work_int'] = ego_papers.paper_id.str[1:].astype('int64')
WORK_KEY = sorted(ego_papers.work_int.tolist())

con.register('ego_w', ego_papers[['work_int']].rename(columns={'work_int': 'work_id'}))
def scan_coauthors():
    return con.sql(f'''SELECT work_id, author_id, any_value(author_display_name) AS author_display_name,
                              string_agg(DISTINCT institution_id, ';') AS institution_ids
                       FROM read_parquet('{npc.AUTHORSHIPS}/bucket=*/*.parquet', hive_partitioning = true)
                       WHERE work_id IN (SELECT work_id FROM ego_w) GROUP BY work_id, author_id''').df()
t0 = time.time()
co_rows = cache('coauthor_rows', scan_coauthors, key=WORK_KEY, sources=AUTH_SOURCES)
TIMINGS['authorship_scan_works'] = round(time.time() - t0, 1)
co_rows['paper_id'] = 'W' + co_rows.work_id.astype('int64').astype(str)
co_rows['author_id_str'] = 'A' + co_rows.author_id.astype('int64').astype(str)
pos = co_rows.groupby('paper_id').agg(n_authors_scan=('author_id', 'nunique')).reset_index()
ego_papers = ego_papers.merge(pos, on='paper_id', how='left')
# works up to YEAR_MAX: OpenAlex publication year, the SciSciNet (MAG) year where OpenAlex has none
con.register('ego_p_all', pd.DataFrame({'paper_id': ego_papers.paper_id.astype('string')}))
wy = cache('work_years', lambda: con.sql(f'''
        SELECT a.paper_id, m.year, s.ssn_year FROM ego_p_all a
        LEFT JOIN (SELECT DISTINCT paper_id, year FROM read_parquet('{npc.OA / 'paper_metadata.parquet'}') WHERE paper_id IN (SELECT paper_id FROM ego_p_all)) m USING (paper_id)
        LEFT JOIN (SELECT paperid AS paper_id, any_value(year) AS ssn_year FROM read_parquet('{npc.SSN_PAPERS}')
                   WHERE paperid IN (SELECT paper_id FROM ego_p_all) GROUP BY 1) s USING (paper_id)''').df(),
           key=sorted(ego_papers.paper_id), sources=[npc.OA / 'paper_metadata.parquet', npc.SSN_PAPERS])
wy = wy.drop_duplicates('paper_id')
wy['year_any'] = pd.to_numeric(wy.year, errors='coerce').fillna(pd.to_numeric(wy.ssn_year, errors='coerce'))
N_WORKS_ALL, N_WORKS_AFTER, N_WORKS_UNDATED = len(ego_papers), int((wy.year_any > YEAR_MAX).sum()), int(wy.year_any.isna().sum())
keep_works = set(wy.loc[wy.year_any <= YEAR_MAX, 'paper_id'])
ego_papers = ego_papers[ego_papers.paper_id.isin(keep_works)].reset_index(drop=True)
co_rows = co_rows[co_rows.paper_id.isin(keep_works)]
ego_rows = ego_rows[ego_rows.paper_id.isin(keep_works)]
WORK_KEY = sorted(ego_papers.work_int.tolist())
print(f'works up to {YEAR_MAX}: {len(ego_papers):,} of {N_WORKS_ALL:,} ({N_WORKS_AFTER:,} later, {N_WORKS_UNDATED:,} without any year — left out)')
EGO_PAPER_COLS = ['paper_id', 'n_authors_scan', 'institution_ids']
names = ego_rows.groupby('author_display_name').size().sort_values(ascending=False)
AUTHOR_NAME = names.index[0] if len(names) else AUTHOR_ID
orcids = sorted(set(ego_rows.orcid.dropna()))
print(f'{len(ego_papers):,} works for {AUTHOR_IDS} ({AUTHOR_NAME})')
if not len(ego_rows):
    print(f'WARNING: no works for {AUTHOR_IDS} in the authorship snapshot (OpenAlex 2026-01). OpenAlex creates new author ids after it '
          '(PrizeAtlas often links such ids): look up the id that carries the works (pqrs, laureate_author_map.csv, OpenAlex).')
print('display names:', names.head(8).to_dict(), '| ORCID:', orcids)

INST_PQ = npc.CACHE / 'institutions.parquet'
inst_csv = npc.OA_RAW / 'institutions.csv.gz'
if inst_csv.exists() and not npc.fresh(INST_PQ, [inst_csv]):
    npc.copy_atomic(con, f'''SELECT replace(id, 'https://openalex.org/', '') AS institution_id, display_name, country_code, type
                             FROM read_csv('{inst_csv}', header = true, all_varchar = true)''', INST_PQ)
if INST_PQ.exists():
    aff = (ego_rows.dropna(subset=['institution_id']).groupby('institution_id').agg(n_works=('work_id', 'nunique')).reset_index()
           .merge(pd.read_parquet(INST_PQ), on='institution_id', how='left').sort_values('n_works', ascending=False))
    display(aff.head(12)); save_table(aff, 'affiliations')

## 2b. Inventor search by name in PatentsView (when pqrs has no inventor)

The pqrs crosswalk covers only the (author, inventor) pairs its matching produced. When it gives no current inventor
id for the author (`NAME_SEARCH = 'auto'`; `'always'` runs it anyway and adds what it selects, `'off'` skips it),
`np_common.inventor_name_search` looks for the person directly in `PatentView/Granted/g_inventor_disambiguated.tsv.zip`
(read from its parquet copy in `cache/`, rebuilt when the zip changes; 2025-12-31 release, so no id translation):

* **name**: the inventor's last name contains the author's family name and the first names agree (equal, or an
  initial), without conflicting middle initials; every display name the author uses on at least 5 % of the works;
* **evidence** that a candidate is the same person, since names alone pull in homonyms:
  co-inventors whose name equals a co-author's (works with at most `MAX_TEAM_SIZE` authors), patents citing the
  author's works (Reliance on Science), assignees sharing a distinctive word with the author's affiliations
  (`DeepMind Technologies Limited` ~ `DeepMind (United Kingdom)`), inventor countries among the affiliation countries,
  patent years overlapping the publication years (5 years before the first to 15 after the last), and how many inventor
  ids carry the same first + last name;
* **selected** when the years overlap and there is a co-author co-inventor, or patents citing the works together with
  an assignee, country or rare-name signal, or an affiliation assignee together with a rare name (at most 2 ids).

Every candidate and its evidence are in `inventor_name_search.csv`; `INVENTOR_IDS` takes any of them by hand.
Checked against pqrs on three laureates: the search selects exactly the pqrs ids of Hassabis, Doudna (50 co-author
co-inventors, 137 patents citing her works) and Horvitz, and finds no inventor named like Serge Haroche.

In [ ]:
name_cands = pd.DataFrame()
INVENTOR_SOURCE = 'explicit INVENTOR_IDS' if INVENTOR_IDS else ('pqrs' if inv_sel else 'none')
if NAME_SEARCH == 'always' or (NAME_SEARCH == 'auto' and not inv_sel and not INVENTOR_IDS):
    variants = [n for n, k in names.items() if k >= max(2, .05 * names.sum())] or list(names.index[:1])
    small_works = set(ego_papers.loc[ego_papers.n_authors_scan <= MAX_TEAM_SIZE, 'paper_id'])
    co_names = sorted(co_rows.loc[~co_rows.author_id_str.isin(AUTHOR_IDS) & co_rows.paper_id.isin(small_works), 'author_display_name'].dropna().unique())
    all_works = sorted(int(p[1:]) for p in wy.paper_id)      # every work, also after YEAR_MAX: citations of any of them are evidence
    aff_names = aff.display_name.dropna().tolist() if 'aff' in globals() else []
    aff_ctry = aff.country_code.dropna().tolist() if 'aff' in globals() else []
    t0 = time.time()
    name_cands = cache('inventor_name_search',
                       lambda: npc.inventor_name_search(con, variants, co_names, all_works, aff_names, aff_ctry, wy.year_any.dropna().tolist()),
                       key=[variants, co_names, all_works, aff_names, aff_ctry],
                       sources=[npc.CACHE / 'g_inventor_disambiguated.parquet', npc.CACHE / 'g_assignee_disambiguated.parquet', npc.PCS_PQ])
    TIMINGS['inventor_name_search'] = round(time.time() - t0, 1)
    print(f'name variants searched: {variants}; {len(co_names):,} co-author names; {len(aff_names)} affiliations')
    display(name_cands.drop(columns=['patent_ids']).head(15))
    save_table(name_cands, 'inventor_name_search')
    picked = name_cands.loc[name_cands.selected.astype(bool), 'inventor_id'].tolist() if len(name_cands) else []
    added = [i for i in picked if i not in inv_sel]
    if added:
        inv_sel = list(dict.fromkeys(inv_sel + added))
        sel_rule = f'{sel_rule} + PatentsView name search' if sel_pqrs else 'PatentsView name search (co-author / citation / assignee evidence)'
        INVENTOR_SOURCE = 'pqrs + name search' if INVENTOR_SOURCE == 'pqrs' else 'name search'
        print(f'added from the name search: {added}')
    elif len(name_cands):
        print('no candidate passes the evidence rule; set INVENTOR_IDS to one of the inventor ids above to use it')
else:
    print(f'name search not run (NAME_SEARCH={NAME_SEARCH!r}; pqrs gave {len(inv_sel)} current inventor id(s))')
print(f'inventor ids used: {inv_sel} [{INVENTOR_SOURCE}]')

## 3. The inventor's patents

All patents of the selected current inventor ids (`g_inventor_disambiguated`, 2025-12-31 release) with
title, grant date and type. Only **utility** patents carry metrics in the PatentView pipeline;
Inventor-position roles are not analysed.

In [ ]:
if inv_sel:
    ego_patents = con.sql(f'''
        SELECT i.patent_id, i.inventor_id, i.inventor_sequence, i.name_first, i.name_last, g.patent_type, g.patent_date,
               year(g.patent_date) AS grant_year_raw, g.patent_title, g.num_claims
        FROM inv i LEFT JOIN gpat g USING (patent_id)
        WHERE i.inventor_id IN ({npc.sql_list(inv_sel)})
        QUALIFY row_number() OVER (PARTITION BY i.patent_id ORDER BY i.inventor_sequence, i.inventor_id) = 1
        ORDER BY g.patent_date, i.patent_id''').df()
else:
    ego_patents = pd.DataFrame(columns=['patent_id', 'inventor_id', 'inventor_sequence', 'name_first', 'name_last', 'patent_type', 'patent_date',
                                        'grant_year_raw', 'patent_title', 'num_claims'])
if PATENT_YEAR_BASIS == 'filing':
    fyr = con.sql(f"SELECT patent_id, filing_year FROM read_parquet('{npc.PV / 'patent_metadata.parquet'}') WHERE patent_id IN ({npc.sql_list(ego_patents.patent_id)})").df()
    ego_patents['basis_year'] = pd.to_numeric(ego_patents.patent_id.map(fyr.set_index('patent_id').filing_year), errors='coerce')
else:
    ego_patents['basis_year'] = pd.to_numeric(ego_patents.grant_year_raw, errors='coerce')
N_PATENTS_ALL = len(ego_patents)
ego_patents = ego_patents[ego_patents.basis_year <= YEAR_MAX].reset_index(drop=True)
print(f'patents with {PATENT_YEAR_BASIS} year up to {YEAR_MAX}: {len(ego_patents):,} of {N_PATENTS_ALL:,}')
INVENTOR_NAME = (ego_patents.name_first + ' ' + ego_patents.name_last).mode().iloc[0] if len(ego_patents) else None
print(f'{len(ego_patents):,} patents for {inv_sel} ({INVENTOR_NAME}); by type {ego_patents.patent_type.value_counts().to_dict()}')
# id tables are registered with a string dtype: an empty object column would register as INTEGER in DuckDB
con.register('ego_t_all', pd.DataFrame({'patent_id': ego_patents.patent_id.astype('string')}))
util_ids = pd.DataFrame({'patent_id': ego_patents.loc[ego_patents.patent_type.eq('utility'), 'patent_id'].astype('string')})
con.register('ego_t', util_ids)
display(ego_patents[['patent_id', 'patent_type', 'patent_date', 'patent_title', 'inventor_sequence']].head(8))

## 4. Work metrics

`np_common.assemble_paper_metrics` attaches every Science of Science paper output (cached per author);
`finish_paper_metrics` adds DA from the shared population table, the CD percentiles among papers with
references, the no-reference and open-window masks and the zero-fills. Works missing from the 2026-01
OpenAlex works table keep their id and their patent-citation total (`pcs_C_total`) but no other metric;
`year_display` falls back to the SciSciNet (MAG) year for them, for timelines only.

In [ ]:
con.register('ego_p', pd.DataFrame({'paper_id': ego_papers.paper_id.astype('string')}))
t0 = time.time()
raw = cache('papers_metrics_raw', lambda: npc.assemble_paper_metrics(con, 'ego_p', with_sciscinet=True),
            key=sorted(ego_papers.paper_id), sources=PAPER_SOURCES)
papers = ego_papers[EGO_PAPER_COLS].merge(raw, on='paper_id', how='left')
papers = npc.finish_paper_metrics(con, papers, 'ego_p', cd_windows=('5', '10'), require_refs=CD_REQUIRE_REFS, obs_end=WINDOW_END_YEAR)
papers = npc.add_paper_percentiles(con, papers, 'ego_p')                 # shared population tables (built once)
papers = npc.mask_open_windows(papers, 'year', WINDOW_END_YEAR, npc.paper_window_cols)
papers['in_metrics'] = papers.research if RESEARCH_ONLY else papers.has_metadata
TIMINGS['paper_metrics'] = round(time.time() - t0, 1)
PAPER_MEASURES = [   # key, label, mid-rank percentile, ties-inclusive percentile (top shares)
    ('impact5', 'Impact: citations, 5 years', 'C_5_pctl', 'C_5_pctl_cume'),
    ('impact10', 'Impact: citations, 10 years', 'C_10_pctl', 'C_10_pctl_cume'),
    ('techlink', 'Cited by patents within 5 years', 'pcs_C_5_pctl', 'pcs_C_5_pctl_cume'),
    ('disruption', 'Disruption (CD, 5 years)', 'CD_5_rpctl_mid', 'CD_5_rpctl_cume'),
    ('F', 'Foundation share (5 years)', 'F_5_pctl', 'F_5_pctl_cume'),
    ('E', 'Extension share (5 years)', 'E_5_pctl', 'E_5_pctl_cume'),
    ('G', 'Generalization share (5 years)', 'G_5_pctl', 'G_5_pctl_cume'),
    ('novelty', 'Novelty (atypical pairs, low Z_10pct)', 'novelty_pctl', 'novelty_pctl_cume'),
    ('conventionality', 'Conventionality (Z_median)', 'Z_median_pctl', 'Z_median_pctl_cume'),
    ('DA', 'Discursive atypicality (DA)', 'DA_pctl', 'DA_pctl'),
    ('SB', 'Sleeping beauty (SB_B)', 'SB_B_pctl', 'SB_B_pctl_cume'),
]
R = papers[papers.in_metrics]
print(f'{len(papers):,} works: {int(papers.has_metadata.sum()):,} with OpenAlex metadata, {int(papers.in_metrics.sum()):,} in the headline statistics '
      f'({"research doctypes" if RESEARCH_ONLY else "all with metadata"}); doctypes {papers.doctype.value_counts(dropna=False).to_dict()}')
print(f'CD blanked for {int(papers.cd_no_refs.sum())} works without references; DA: {int(papers.da_scored.sum())} scored, '
      f'{int(papers.DA_pctl.notna().sum())} in the article population (census years {sorted(papers.da_year.dropna().astype(int).unique().tolist())})')

nometa = papers[~papers.has_metadata].sort_values('pcs_C_total', ascending=False)
nm_share = float(nometa.pcs_C_total.sum() / max(1, papers.pcs_C_total.sum()))
print(f'{len(nometa)} works have no OpenAlex metadata; they hold {int(nometa.pcs_C_total.sum()):,} of {int(papers.pcs_C_total.sum()):,} '
      f'patent citations ({nm_share:.0%})' + ('  WARNING: a large share of the record has no metadata' if nm_share > .05 else ''))
display(nometa[['paper_id', 'year_display', 'ssn_doi', 'ssn_C10', 'pcs_C_total']].head(8))
save_table(nometa[['paper_id', 'year_display', 'year_display_source', 'ssn_doi', 'ssn_C5', 'ssn_C10', 'pcs_C_total', 'n_authors_scan']], 'works_without_metadata')
display(R.sort_values('C_5_pctl', ascending=False)[['paper_id', 'year', 'doctype', 'journal'] + [m[2] for m in PAPER_MEASURES]].head(12).round(3))

## 5. Patent metrics

The PatentView outputs for the utility patents (cached per author, keyed on the patent set), then:
zero-fills for never-cited patents (`patent_citation` lists only cited ones), the no-reference CD mask,
the open-window mask on grant year, DA from the shared population table (filing year × CPC Section),
and a `metrics_note` for patents that lack a metric family. Patent citation percentiles are within WIPO
sector × grant year (SoS); the percentiles of every patent measure used below are within grant year × CPC Section.

In [ ]:
def assemble_patents():
    # runs also for an author without patents: the joins then return empty frames with every column
    T = util_ids[['patent_id']].astype(object)
    k = dict(key='patent_id', table='ego_t')
    T = T.merge(npc.semi(con, npc.PV / 'patent_metadata.parquet', ['grant_year', 'filing_year', 'ref_count', 'cpc_code', 'cpc_code_list', 'assignee_list'], **k), how='left')
    cit = [f'{c}_{w}' for w in npc.WINDOWS for c in ('C', 'appC', 'uniqueC')] + [f'C_{s}_{w}' for w in npc.WINDOWS for s in ('examiner', 'non_examiner', 'unknown')]
    T = T.merge(npc.semi(con, npc.PV / 'patent_citation.parquet', cit, **k), how='left')
    T = T.merge(npc.semi(con, npc.PV / 'patent_hit_probability.parquet', ['wipo_sector'] + [f'pctl_uniqueC_{w}' for w in npc.WINDOWS] +
                         ['pctl_C_5', 'pctl_C_all', 'pctl_appC_5'], **k), how='left')
    T = T.merge(npc.semi(con, npc.PV / 'patent_disruption.parquet', npc.CD_COLS + npc.CD_PCTL_COLS + ['pctl_year', 'pctl_group'], **k), how='left')
    T = T.merge(npc.semi(con, npc.PV / 'patent_z_score.parquet', ['Z_median', 'Z_10pct', 'Z_min', 'n_pairs'], **k), how='left')
    T = T.merge(npc.semi(con, npc.PV / 'patent_sb.parquet', ['SB_B', 'SB_T', 'n_cite'], **k), how='left')
    T = T.merge(npc.semi(con, npc.PV / 'patent_inventor_country.parquet', ['n_inventors', 'n_located', 'n_countries', 'countries', 'is_international',
                                                                           'first_inventor_country', 'assignee_countries'], **k), how='left')
    ppp = con.sql(f'''SELECT patent_id, any_value(tech_DA_year_cohort) AS ppp_tech_DA_year, any_value(tech_DA_year_section_cohort) AS ppp_tech_DA_year_section,
                             count(*) AS n_ppp_pairs, count(*) FILTER (WHERE coalesce(pap_id_mislink_suspected, false)) AS n_ppp_pairs_mislink,
                             string_agg(DISTINCT paper_id, ';') AS ppp_paper_ids
                      FROM read_parquet('{npc.PPP_MASTER}') WHERE patent_id IN (SELECT patent_id FROM ego_t) GROUP BY patent_id''').df()
    return T.merge(ppp, on='patent_id', how='left')

t0 = time.time()
raw_t = cache('patents_metrics_raw', assemble_patents, key=sorted(util_ids.patent_id), sources=PATENT_SOURCES)
patents = ego_patents.merge(raw_t, on='patent_id', how='left')
for c in ['grant_year', 'filing_year']:
    patents[c] = pd.to_numeric(patents[c], errors='coerce').astype('Int64') if c in patents else pd.Series(pd.NA, index=patents.index, dtype='Int64')
cnt = [f'{c}_{w}' for w in npc.WINDOWS for c in ('C', 'appC', 'uniqueC')] + [f'C_{s}_{w}' for w in npc.WINDOWS for s in ('examiner', 'non_examiner', 'unknown')]
for c in [c for c in cnt if c in patents]:
    patents[c] = pd.to_numeric(patents[c], errors='coerce').fillna(0).astype('Int64').where(patents.grant_year.notna())
patents['cpc_section'] = patents.get('cpc_code', pd.Series(pd.NA, index=patents.index)).astype('string').str[:1]
patents['in_ppp'] = patents.get('n_ppp_pairs', pd.Series(np.nan, index=patents.index)).notna()
patents = npc.add_patent_da(con, patents, 'ego_t')
if CD_REQUIRE_REFS and 'ref_count' in patents:
    patents = npc.mask_cd_without_refs(patents, year_col='grant_year')
for w in npc.WINDOWS:
    if f'CD_{w}_pctl' in patents:
        patents[f'CD_{w}_pctl_mid'] = (patents[f'CD_{w}_pctl'] + patents[f'CD_{w}_pctl_cume']) / 2
patents = npc.add_patent_percentiles(con, patents, 'ego_t', OBS_END_YEAR)   # shared population table (built once)
patents = npc.mask_open_windows(patents, 'grant_year', WINDOW_END_YEAR, npc.patent_window_cols)
patents['metrics_note'] = np.select(
    [~patents.patent_type.eq('utility'), patents.grant_year.isna(), patents.pppl.isna(), patents.cpc_section.isna(), patents.get('cd_no_refs', False) == True],
    ['not a utility patent: no metrics', 'not in patent_metadata', 'not scored for DA', 'no CPC Section: no DA, CD percentile, Kim z',
     'no references: CD family blanked'], '')
TIMINGS['patent_metrics'] = round(time.time() - t0, 1)
UT = patents[patents.grant_year.notna()]
print(f'{len(patents):,} patents; {len(UT):,} utility patents with metadata; CD for {int(UT.CD_5.notna().sum()) if "CD_5" in UT else 0}, '
      f'Kim z for {int(UT.Z_median.notna().sum()) if "Z_median" in UT else 0}, DA for {int(UT.DA_pctl.notna().sum())}, in PPP {int(patents.in_ppp.sum())}')
print('notes:', patents.metrics_note.replace('', 'ok').value_counts().to_dict())
PATENT_MEASURES = [
    ('impact5', 'Impact: citations, 5 years (uniqueC)', 'uniqueC_5_pctl', 'uniqueC_5_pctl_cume'),
    ('techlink', 'Science linkage: papers cited', 'n_paper_refs_pctl', 'n_paper_refs_pctl_cume'),
    ('disruption', 'Disruption (CD, 5 years)', 'CD_5_rpctl_mid', 'CD_5_rpctl_cume'),
    ('F', 'Foundation share (5 years)', 'F_5_pctl', 'F_5_pctl_cume'),
    ('E', 'Extension share (5 years)', 'E_5_pctl', 'E_5_pctl_cume'),
    ('G', 'Generalization share (5 years)', 'G_5_pctl', 'G_5_pctl_cume'),
    ('novelty', 'Novelty (atypical CPC pairs, low Z_10pct)', 'novelty_pctl', 'novelty_pctl_cume'),
    ('conventionality', 'Conventionality (Z_median)', 'Z_median_pctl', 'Z_median_pctl_cume'),
    ('DA', 'Discursive atypicality (DA)', 'DA_pctl', 'DA_pctl'),
    ('SB', 'Sleeping beauty (SB_B)', 'SB_B_pctl', 'SB_B_pctl_cume'),
]
RAW_COL = {'works': {'C_5_pctl': 'C_5', 'C_10_pctl': 'C_10', 'pcs_C_5_pctl': 'pcs_C_5', 'CD_5_rpctl_mid': 'CD_5', 'F_5_pctl': 'F_5', 'E_5_pctl': 'E_5',
                     'G_5_pctl': 'G_5', 'novelty_pctl': 'Z_10pct', 'Z_median_pctl': 'Z_median', 'DA_pctl': 'DA_log', 'SB_B_pctl': 'SB_B'},
           'patents': {'uniqueC_5_pctl': 'uniqueC_5', 'n_paper_refs_pctl': 'n_paper_refs', 'CD_5_rpctl_mid': 'CD_5', 'F_5_pctl': 'F_5', 'E_5_pctl': 'E_5',
                       'G_5_pctl': 'G_5', 'novelty_pctl': 'Z_10pct', 'Z_median_pctl': 'Z_median', 'DA_pctl': 'DA_log', 'SB_B_pctl': 'SB_B'}}
show = [c for c in ['patent_id', 'grant_year', 'filing_year', 'patent_title', 'cpc_section'] + [m[2] for m in PATENT_MEASURES if m[2] in patents] + ['metrics_note']
        if c in patents]
display(patents.sort_values('uniqueC_5_pctl', ascending=False)[show].head(12).round(3) if len(patents) else patents.head())

## 6. Patent ↔ paper links

1. **Patents citing the person's works** (Reliance on Science). The PCS patent numbers are normalised
   to PatentsView ids (utility digits, `RE…`, `D…`, `PP…` incl. pre-2001 plant `-p` kinds, `H` padded to
   6 digits; non-US rows dropped). Pre-grant publications (kinds a1/a2/a9/p1) are kept as `doc_kind =
   pgpub`, mapped to their granted patent when one exists, and each citing **invention**
   (`coalesce(granted patent, pgpub)`) is counted once per work. `self_citation` = the citing invention
   is the person's own patent, or a pre-grant publication naming one of the person's inventor ids.
2. **Science references of the person's patents** — `n_frontpage` counts front-page references
   (`frontonly` or `both`), `n_body` in-text ones (`bodyonly` or `both`).
3. **PPP pairs** from `ppp_master_rev` touching the person; `pair_type = self` when both sides are
   theirs. PPP-corpus DA columns are prefixed `ppp_`; the population DA percentile of each side is
   shown next to them. Pairs flagged `pap_id_mislink_suspected` are counted but reported.

In [ ]:
PCS_PQ = npc.CACHE / 'pcs_oa_uspto.v2.parquet'
if not npc.fresh(PCS_PQ, [npc.PCS_CSV]):
    t1 = time.time()
    npc.copy_atomic(con, f'''
        WITH r AS (SELECT reftype, TRY_CAST(confscore AS INTEGER) AS confscore, TRY_CAST(oaid AS BIGINT) AS oaid, patent, wherefound,
                          regexp_matches(patent, '^us-[a-z]*[0-9]+-[a-z0-9]+$') AS ok,
                          regexp_extract(patent, '^us-([a-z]*)[0-9]+-', 1) AS pre,
                          regexp_extract(patent, '^us-[a-z]*([0-9]+)-', 1) AS digits,
                          regexp_extract(patent, '-([a-z0-9]+)$', 1) AS kind
                   FROM read_csv('{npc.PCS_CSV}', header = true, all_varchar = true))
        SELECT reftype, confscore, oaid, patent, wherefound, kind,
               CASE WHEN NOT ok OR pre NOT IN ('', 're', 'd', 'pp', 'h') THEN NULL
                    WHEN kind IN ('a1', 'a2', 'a9', 'p1') THEN 'pgpub' ELSE 'grant' END AS doc_kind,
               CASE WHEN NOT ok OR pre NOT IN ('', 're', 'd', 'pp', 'h') THEN NULL          -- us-dk-... (non-US) and odd prefixes
                    WHEN kind IN ('a1', 'a2', 'a9', 'p1') THEN substr(digits, 1, 4) || lpad(substr(digits, 5), 7, '0')
                    WHEN pre = '' AND kind LIKE 'p%' THEN 'PP' || ltrim(digits, '0')
                    WHEN pre = 'h' THEN 'H' || lpad(ltrim(digits, '0'), 6, '0')
                    WHEN pre = '' THEN ltrim(digits, '0')
                    ELSE upper(pre) || ltrim(digits, '0') END AS patent_id
        FROM r''', PCS_PQ)
    print(f'PCS csv -> parquet in {time.time() - t1:.0f}s')
con.execute(f"CREATE OR REPLACE VIEW pcs AS SELECT * FROM read_parquet('{PCS_PQ}')")
con.register('ego_pi', pd.DataFrame({'oaid': ego_papers.work_int.astype('int64')}))

raw_in = cache('pcs_citing_rows', lambda: con.sql('SELECT * FROM pcs WHERE oaid IN (SELECT oaid FROM ego_pi)').df(),
               key=WORK_KEY, sources=[PCS_PQ])
con.register('raw_in', raw_in)
cites_in = con.sql('''
    SELECT 'W' || CAST(r.oaid AS VARCHAR) AS paper_id, r.patent_id, r.doc_kind, r.reftype, r.confscore, r.wherefound,
           CASE WHEN r.doc_kind = 'pgpub' THEN x.patent_id ELSE r.patent_id END AS granted_patent_id,
           g.patent_date, g.patent_title, g.patent_type
    FROM raw_in r
    LEFT JOIN (SELECT pgpub_id, min(patent_id) AS patent_id FROM pgx GROUP BY 1) x ON r.doc_kind = 'pgpub' AND x.pgpub_id = r.patent_id
    LEFT JOIN gpat g ON g.patent_key = regexp_replace(CASE WHEN r.doc_kind = 'pgpub' THEN x.patent_id ELSE r.patent_id END, '^H0+', 'H')
    WHERE r.patent_id IS NOT NULL''').df()
cites_in['invention_id'] = cites_in.granted_patent_id.fillna(cites_in.patent_id)
own_pgpubs = set(con.sql(f"SELECT DISTINCT pgpub_id FROM pginv WHERE inventor_id IN ({npc.sql_list(inv_sel)})").df().pgpub_id) if inv_sel else set()
cites_in['self_citation'] = cites_in.invention_id.isin(set(ego_patents.patent_id)) | cites_in.patent_id.isin(own_pgpubs)
cites_in['cite_year'] = pd.to_datetime(cites_in.patent_date).dt.year.astype('Int64')
cites_in = cites_in.merge(papers[['paper_id', 'year_display']].rename(columns={'year_display': 'paper_year'}), on='paper_id', how='left')
per_paper = cites_in.groupby('paper_id').agg(n_citing_inventions=('invention_id', 'nunique'),
                                             n_citing_self=('invention_id', lambda s: s[cites_in.loc[s.index, 'self_citation']].nunique()))
papers = papers.drop(columns=[c for c in per_paper.columns if c in papers]).merge(per_paper, on='paper_id', how='left')
papers[['n_citing_inventions', 'n_citing_self']] = papers[['n_citing_inventions', 'n_citing_self']].fillna(0).astype(int)
print(f'{len(cites_in):,} PCS rows to {cites_in.paper_id.nunique():,} works: {int((cites_in.doc_kind == "grant").sum()):,} from granted patents, '
      f'{int((cites_in.doc_kind == "pgpub").sum()):,} from pre-grant publications ({int(cites_in.loc[cites_in.doc_kind == "pgpub", "granted_patent_id"].notna().sum()):,} of them granted later); '
      f'{cites_in.invention_id.nunique():,} distinct citing inventions, {cites_in.loc[cites_in.self_citation, "invention_id"].nunique():,} of them the person\'s own; '
      f'examiner rows {int((cites_in.reftype == "exm").sum())}')
save_table(cites_in.sort_values(['paper_id', 'patent_date']), 'patents_citing_papers')
display(cites_in.groupby('invention_id').agg(n_works_cited=('paper_id', 'nunique'), date=('patent_date', 'first'), title=('patent_title', 'first'),
                                             self_citation=('self_citation', 'max')).sort_values('n_works_cited', ascending=False).head(8))

# science references of the person's patents
raw_out = cache('pcs_patent_refs', lambda: con.sql('SELECT * FROM pcs WHERE patent_id IN (SELECT patent_id FROM ego_t_all)').df(),
                key=sorted(ego_patents.patent_id), sources=[PCS_PQ])
refs_out = raw_out.assign(cited_paper_id='W' + raw_out.oaid.astype('Int64').astype(str))
refs_out['self_reference'] = refs_out.cited_paper_id.isin(set(papers.paper_id))
ref_sum = refs_out.groupby('patent_id').agg(n_paper_refs=('cited_paper_id', 'nunique'), mean_confscore=('confscore', 'mean'),
                                            n_frontpage=('wherefound', lambda s: int(s.isin(['frontonly', 'both']).sum())),
                                            n_body=('wherefound', lambda s: int(s.isin(['bodyonly', 'both']).sum())))
ref_sum = ref_sum.join(refs_out[refs_out.self_reference].groupby('patent_id').cited_paper_id.nunique().rename('n_self_paper_refs')).reset_index()
patents = patents.drop(columns=[c for c in ref_sum.columns if c != 'patent_id' and c in patents]).merge(ref_sum, on='patent_id', how='left')
for c in ['n_paper_refs', 'n_self_paper_refs', 'n_frontpage', 'n_body']:
    patents[c] = patents[c].fillna(0).astype(int)
print(f'{len(refs_out):,} science references from {refs_out.patent_id.nunique():,} of the person\'s patents; {int(refs_out.self_reference.sum()):,} to their own works')
save_table(refs_out.drop(columns=['oaid']), 'patents_science_references')
save_parquet(patents, 'patents_metrics'); save_table(patents, 'patents_metrics')

# PPP pairs
PPP_COLS = ['paper_id', 'patent_id', 'pub_year', 'filing_year', 'grant_year', 'ppp_score', 'sci_DA_year_cohort', 'sci_DA_year_field_cohort',
            'tech_DA_year_cohort', 'tech_DA_year_section_cohort', 'pap_C_5', 'pap_C_all', 'pap_pctl_c5', 'pap_CD_5', 'pap_F_5', 'pap_E_5', 'pap_G_5',
            'pat_uniqueC_5', 'pat_uniqueC_all', 'pat_pctl_uniqueC_5', 'pat_CD_5', 'pat_F_5', 'pat_E_5', 'pat_G_5', 'pap_FoS_0', 'cpc_section',
            'pat_assignee_list', 'pap_id_mislink_suspected']
ppp = con.sql(f'''SELECT {', '.join(PPP_COLS)}, paper_id IN (SELECT paper_id FROM ego_p) AS paper_is_own, patent_id IN (SELECT patent_id FROM ego_t_all) AS patent_is_own
                  FROM read_parquet('{npc.PPP_MASTER}')
                  WHERE paper_id IN (SELECT paper_id FROM ego_p) OR patent_id IN (SELECT patent_id FROM ego_t_all)''').df()
ppp = ppp.rename(columns={'sci_DA_year_cohort': 'ppp_sci_DA_year', 'sci_DA_year_field_cohort': 'ppp_sci_DA_year_field',
                          'tech_DA_year_cohort': 'ppp_tech_DA_year', 'tech_DA_year_section_cohort': 'ppp_tech_DA_year_section'})
for c in [c for c in ppp.columns if re.fullmatch(r'pat_(C|appC|uniqueC)_(3|5|10|all)', c)]:
    ppp[c] = ppp[c].fillna(0).where(ppp.grant_year.notna())
ppp['pair_type'] = np.select([ppp.paper_is_own & ppp.patent_is_own, ppp.paper_is_own], ['self', 'paper_only'], 'patent_only')
ppp = ppp.merge(papers[['paper_id', 'DA_pctl']].rename(columns={'DA_pctl': 'pap_DA_pctl_pop'}), on='paper_id', how='left') \
         .merge(patents[['patent_id', 'DA_pctl']].rename(columns={'DA_pctl': 'pat_DA_pctl_pop'}), on='patent_id', how='left')
print(f'{len(ppp):,} PPP pairs touching the person:', ppp.pair_type.value_counts().to_dict())
print('mislink-suspected pairs by type:', ppp.groupby(['pair_type', ppp.pap_id_mislink_suspected.fillna(False)]).size().to_dict())
save_table(ppp, 'ppp_pairs')

## 6b. Textbook reach: citations from books

A research result that enters textbooks, monographs and encyclopedias is cited by **books**. Every work is
matched against the OpenAlex citation graph (`referenced_works_w_year`): the citing works whose OpenAlex type
is `book`, `book-chapter`, `book-section` or `reference-entry`. For each work: the number of distinct citing
books / chapters / entries (`n_book_cites`, split by type), the first and last citing year, and the
**book-citation percentile** within publication year × first FoS over all papers (most papers are never cited
by a book and share the bottom tie block). Works cited by at least `BOOK_MIN` books are counted as
"textbook reach". OpenAlex types textbooks and monographs alike as `book`, so this measures reach into the
book literature rather than textbooks alone. With `FETCH_TITLES` the titles of the works and of the citing
books are added.

In [ ]:
t0 = time.time()
papers = npc.add_book_citations(con, papers, 'ego_p')            # shared population tables (built once, a few minutes)
TIMINGS['book_citations'] = round(time.time() - t0, 1)
if not any(m[0] == 'books' for m in PAPER_MEASURES):
    PAPER_MEASURES.append(('books', 'Cited by books (textbook reach)', 'book_cites_pctl', 'book_cites_pctl_cume'))
RAW_COL['works']['book_cites_pctl'] = 'n_book_cites'

def scan_citing_books():
    types = npc.sql_list(npc.BOOK_TYPES)
    return con.sql(f'''
        SELECT r.referenced_work_id AS paper_id, r.work_id AS book_id, r.work_year AS book_year, m.doctype AS book_type, m.journal AS book_source
        FROM read_parquet('{npc.REFS_W_YEAR}/*.parquet') r
        JOIN (SELECT paper_id AS work_id, doctype, journal FROM read_parquet('{npc.OA / 'paper_metadata.parquet'}') WHERE doctype IN ({types})) m
          USING (work_id)
        WHERE r.referenced_work_id IN (SELECT paper_id FROM ego_p)''').df()
t0 = time.time()
books = cache('citing_books', scan_citing_books, key=sorted(ego_papers.paper_id), sources=[npc.OA / 'paper_metadata.parquet'])
TIMINGS['citing_books_scan'] = round(time.time() - t0, 1)

# optional titles of the works and of the citing books (one scan of the raw OpenAlex works table)
if FETCH_TITLES:
    ids = pd.DataFrame({'paper_id': pd.Series(sorted(set(ego_papers.paper_id) | set(books.book_id)), dtype='string')})
    con.register('title_ids', ids)
    if TITLES_TABLE == 'api':                                 # one person: the OpenAlex API is faster than a 205 GB scan
        titles = pd.DataFrame({'paper_id': pd.Series(dtype=str), 'title': pd.Series(dtype=str)})
    elif TITLES_TABLE and not Path(TITLES_TABLE).exists():   # never fall back to a 205 GB scan per author in a batch
        print(f'WARNING: {TITLES_TABLE} not found: titles from the OpenAlex API only')
        titles = pd.DataFrame({'paper_id': pd.Series(dtype=str), 'title': pd.Series(dtype=str)})
    else:
        src_t = f"read_parquet('{TITLES_TABLE}')" if TITLES_TABLE else f"read_parquet('{npc.OA_RAW}/works/works/*.parquet')"
        titles = cache('titles', lambda: con.sql(f'''SELECT id AS paper_id, any_value(title) AS title FROM {src_t}
                                                    WHERE id IN (SELECT paper_id FROM title_ids) GROUP BY 1''').df(),
                       key=sorted(ids.paper_id.astype(str)))
    tmap = titles.set_index('paper_id').title
    papers['title'] = papers.paper_id.map(tmap)
    missing = papers.loc[papers.title.isna(), 'paper_id'].tolist()   # works not in the snapshot's works table: OpenAlex API (cached)
    if missing:
        at = npc.api_titles(missing)
        papers['title'] = papers.title.fillna(papers.paper_id.map(at))
        print(f'titles from the OpenAlex API for {len(at)} of {len(missing)} works missing from the snapshot')
    books['book_title'] = books.book_id.map(tmap)
    TIMINGS['titles'] = TIMINGS.get('titles', 0)

bw = papers[papers.has_metadata]
n_any, n_text = int((bw.n_book_cites > 0).sum()), int((bw.n_book_cites >= BOOK_MIN).sum())
print(f'{len(books):,} citations from {books.book_id.nunique():,} books / chapters / reference entries to {books.paper_id.nunique():,} works')
print(f'works cited by at least one book: {n_any} of {len(bw)} with metadata ({n_any / max(1, len(bw)):.0%}); by at least {BOOK_MIN}: {n_text} '
      f'("textbook reach"); median book-citation percentile {bw.book_cites_pctl.median():.3f}; in the cohort top 10 % / 1 %: '
      f'{npc.share_at_least(bw.book_cites_pctl, .9):.2f} / {npc.share_at_least(bw.book_cites_pctl, .99):.2f}')
print('citing documents by type:', books.drop_duplicates('book_id').book_type.value_counts().to_dict())
cols = ['paper_id', 'year', 'journal', 'doctype', 'C_all', 'n_book_cites', 'n_books', 'n_chapters', 'n_reference_entries',
        'first_book_cite_year', 'last_book_cite_year', 'book_cites_pctl'] + (['title'] if 'title' in papers else [])
book_tab = papers[papers.n_book_cites > 0].sort_values(['n_book_cites', 'C_all'], ascending=False)[cols]
display(book_tab.head(15).round(3))
save_table(book_tab, 'book_citations_by_work')
save_table(books.sort_values(['paper_id', 'book_year']), 'citing_books')
src = books.drop_duplicates('book_id').book_source.fillna('(no source)').value_counts().head(15)
print('most frequent sources of the citing books (publisher series / book title):'); display(src)

# Figure 10: book citations
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
d = bw.dropna(subset=['book_cites_pctl', 'C_5_pctl'])
axes[0].scatter(d.C_5_pctl, d.book_cites_pctl, s=10 + 8 * np.sqrt(d.n_book_cites), color=SC, alpha=.6, lw=0)
axes[0].axhline(.5, color=npc.INK2, lw=.8); axes[0].axvline(.5, color=npc.INK2, lw=.8)
axes[0].set(xlim=(0, 1), ylim=(0, 1), xlabel='impact percentile (5-year citations)', ylabel='book-citation percentile',
            title='Books vs journals (size ∝ citing books)')
top = book_tab.head(15).iloc[::-1]
lbl = (top.title.fillna('').str.slice(0, 45) + ' · ' if 'title' in top else '') + top.year.astype('Int64').astype(str) + ' ' + top.journal.fillna('').str.slice(0, 25)
axes[1].barh(range(len(top)), top.n_book_cites, color=SC, height=.7)
axes[1].set_yticks(range(len(top)), lbl, fontsize=7.5)
axes[1].set(xlabel='citing books / chapters / entries', title=f'Works most cited by books (≥ {BOOK_MIN}: {n_text})')
axes[1].grid(axis='y', visible=False)
per = books.drop_duplicates(['book_id', 'paper_id']).groupby(['book_year', 'book_type']).size().unstack(fill_value=0) if len(books) else pd.DataFrame()
per = per[per.index <= OBS_END_YEAR] if len(per) else per
bottom = np.zeros(len(per))
for t, c_ in zip(['book', 'book-chapter', 'book-section', 'reference-entry'], [SC, LC, LC, GREY]):
    if t in per:
        axes[2].bar(per.index, per[t], bottom=bottom, color=c_, width=.8, label=t, edgecolor='white', linewidth=.5); bottom += per[t].to_numpy()
axes[2].set(xlabel='year of the citing book', ylabel='book citations', title='Book citations per year'); axes[2].legend(loc='upper left', fontsize=8)
savefig(fig, 'fig10_book_citations'); plt.show()

def fig10_html(stacked=False):
    # stacked: one panel per row (the dashboard), else side by side
    pos = [(1, 1), (2, 1), (3, 1)] if stacked else [(1, 1), (1, 2), (1, 3)]
    titles = ['Books vs journals: impact and book-citation percentiles', 'Works most cited by books', 'Book citations per year']
    h = (make_subplots(rows=3, cols=1, subplot_titles=titles, vertical_spacing=.08) if stacked
         else make_subplots(rows=1, cols=3, subplot_titles=titles, column_widths=[.3, .4, .3]))
    h.add_trace(go.Scatter(x=d.C_5_pctl, y=d.book_cites_pctl, mode='markers',
                           marker=dict(color=SC, size=6 + 24 * np.sqrt(d.n_book_cites / max(1, d.n_book_cites.max())), opacity=.6),   # capped bubbles
                           hovertext=doc_hover(d, 'works') + '<br>citing books: ' + d.n_book_cites.astype(int).astype(str)
                                     + '<br>book-citation percentile: ' + d.book_cites_pctl.round(3).astype(str), hoverinfo='text', showlegend=False),
                row=pos[0][0], col=pos[0][1])
    h.add_trace(go.Bar(x=top.n_book_cites, y=lbl, orientation='h', marker_color=SC, showlegend=False,
                       hovertext=top.paper_id + '<br>citing books ' + top.n_book_cites.astype(int).astype(str) + ' (books ' + top.n_books.astype(int).astype(str)
                                 + ', chapters ' + top.n_chapters.astype(int).astype(str) + ', entries ' + top.n_reference_entries.astype(int).astype(str) + ')',
                       hoverinfo='text'), row=pos[1][0], col=pos[1][1])
    for t, c_ in zip(['book', 'book-chapter', 'book-section', 'reference-entry'], [SC, LC, LC, GREY]):
        if len(per) and t in per:
            h.add_bar(x=per.index, y=per[t], name=t, marker_color=c_, row=pos[2][0], col=pos[2][1])
    h.update_xaxes(range=[0, 1], title_text='impact percentile (5-year citations)', row=pos[0][0], col=pos[0][1])
    h.update_yaxes(range=[0, 1], title_text='book-citation percentile', row=pos[0][0], col=pos[0][1])
    h.update_xaxes(title_text='citing books / chapters / entries', row=pos[1][0], col=pos[1][1])
    h.update_xaxes(title_text='year of the citing book', row=pos[2][0], col=pos[2][1]); h.update_yaxes(title_text='book citations', row=pos[2][0], col=pos[2][1])
    h.update_layout(title=f'{AUTHOR_NAME}: citations from books, book chapters and reference entries', barmode='stack', height=1500 if stacked else 560)
    return h
save_html(fig10_html(), 'fig10_book_citations')
dash_only(fig10_html(stacked=True), 'fig10_book_citations')
save_parquet(papers, 'papers_metrics'); save_table(papers, 'papers_metrics')

## 7. Career summary in percentiles

For every measure: the number of documents that have it, the mean and median cohort percentile, and
the shares in the cohort's top 10 % and top 1 % (`percentile_summary.csv`); the yearly mean percentile
of every measure (`yearly_percentiles.csv`); `profile_summary.json` (strict JSON). Figures: output per
year, the percentile profile (one dot per document), percentiles over the career, and impact against
disruption, novelty and DA — each as PNG and as interactive HTML.

In [ ]:
R = papers[papers.in_metrics]                       # refreshed: section 6 added the citing-invention counts
UT = patents[patents.grant_year.notna()]             # refreshed: section 6 added the science-reference counts

def pctl_stats(frame, measures, side):
    rows = []
    for key, label, col, cume in measures:
        # top shares on the same mid-rank percentile: a large tie block (e.g. the never-cited majority) sits at its mid-point,
        # whereas a ties-inclusive threshold would count every uncited document as 'top 1 %'
        v = pd.to_numeric(frame.get(col, pd.Series(dtype=float)), errors='coerce').dropna()
        rows.append({'side': side, 'key': key, 'measure': label, 'column': col, 'n': int(len(v)),
                     'mean_pctl': float(v.mean()) if len(v) else None, 'median_pctl': float(v.median()) if len(v) else None,
                     'share_top10': npc.share_at_least(v, .90) if len(v) else None, 'share_top1': npc.share_at_least(v, .99) if len(v) else None})
    return pd.DataFrame(rows)

PSUM = pd.concat([pctl_stats(R, PAPER_MEASURES, 'works'), pctl_stats(UT, PATENT_MEASURES, 'patents')], ignore_index=True)
save_table(PSUM, 'percentile_summary')
display(PSUM.drop(columns=['column']).round(3))

# yearly mean percentile of every measure (works by publication year, patents by PATENT_YEAR_BASIS year)
UT = UT.assign(basis_year=pd.to_numeric(UT.grant_year if PATENT_YEAR_BASIS == 'grant' else UT.filing_year, errors='coerce'))
yp = R.assign(year=R.year.astype(float)).groupby('year')[[m[2] for m in PAPER_MEASURES]].mean().add_prefix('works_')
yt = UT.groupby('basis_year')[[m[2] for m in PATENT_MEASURES]].mean().add_prefix('patents_') if len(UT) else pd.DataFrame()
cnt = pd.DataFrame({'n_works': R.dropna(subset=['year']).groupby(R.year.astype(float)).size()})
if len(UT):
    cnt = cnt.join(UT.groupby('basis_year').size().rename('n_patents'), how='outer')
yearly = cnt.join(yp, how='outer').join(yt, how='outer').sort_index()
yearly.index = yearly.index.astype(int); yearly.index.name = 'year'
for c in ['n_works', 'n_patents']:
    if c in yearly:
        yearly[c] = yearly[c].fillna(0).astype(int)
save_table(yearly, 'yearly_percentiles', index=True)
display(yearly.tail(8).round(3))

summary = {
    'author_id': AUTHOR_ID, 'author_ids': AUTHOR_IDS, 'author_name': AUTHOR_NAME, 'orcid': orcids, 'year_max': YEAR_MAX,
    'patent_year_basis': PATENT_YEAR_BASIS,
    'pqrs_inventor_ids': sel_pqrs, 'inventor_ids_current': inv_sel, 'inventor_release': [REL_OLD, REL_NEW], 'inventor_name': INVENTOR_NAME,
    'inventor_selection_rule': sel_rule, 'inventor_source': INVENTOR_SOURCE,
    'crosswalk_confidence_selected_min': float(sel_conf.min()) if len(sel_conf) else None,
    'crosswalk_confidence_selected_max': float(sel_conf.max()) if len(sel_conf) else None,
    'n_works_all_years': N_WORKS_ALL, 'n_works_after_year_max': N_WORKS_AFTER, 'n_works_undated': N_WORKS_UNDATED,
    'n_works_record': int(len(papers)), 'n_works_with_metadata': int(papers.has_metadata.sum()), 'n_works_in_metrics': int(len(R)),
    'doctype_counts': papers.doctype.fillna('(no metadata)').value_counts().to_dict(),
    'h_index_all_time_citations': npc.h_index(R.C_all), 'n_works_cd_no_refs': int(R.cd_no_refs.sum()),
    'n_works_without_metadata': int((~papers.has_metadata).sum()), 'pcs_citations_works_without_metadata': int(nometa.pcs_C_total.sum()),
    'citing_inventions_distinct': int(cites_in.invention_id.nunique()), 'citing_inventions_self': int(cites_in.loc[cites_in.self_citation, 'invention_id'].nunique()),
    'n_patents_all_years': N_PATENTS_ALL, 'n_patents': int(len(patents)), 'n_utility_patents_with_metadata': int(len(UT)),
    'n_works_in_ppp': int(papers.in_ppp.sum()), 'n_patents_in_ppp': int(patents.in_ppp.sum()),
    'n_works_cited_by_books': n_any, f'n_works_cited_by_at_least_{BOOK_MIN}_books': n_text, 'book_citations_total': int(len(books)),
    'citing_books_distinct': int(books.book_id.nunique()),
    'n_ppp_pairs': ppp.pair_type.value_counts().to_dict(),
    'percentiles': {side: {r.measure: {k: getattr(r, k) for k in ('n', 'mean_pctl', 'median_pctl', 'share_top10', 'share_top1')}
                           for r in PSUM[PSUM.side == side].itertuples()} for side in ('works', 'patents')},
}
npc.write_json(OUT / 'profile_summary.json', summary); WRITTEN.append(OUT / 'profile_summary.json')
flat = {k: v for k, v in npc.clean_json(summary).items() if not isinstance(v, (list, dict))}
pd.Series(flat).to_frame('value').to_csv(OUT / 'profile_summary.csv'); WRITTEN.append(OUT / 'profile_summary.csv')
display(pd.Series(flat).to_frame('value'))

In [ ]:
# ---------------- Figure 1: cumulative output ----------------
allp = papers.dropna(subset=['year']).assign(year=lambda x: x.year.astype(int), kind=lambda x: np.where(x.in_metrics, 'research', 'other'))
cnt_p = allp.groupby(['year', 'kind']).size().unstack(fill_value=0).reindex(columns=['research', 'other'], fill_value=0)
if len(cnt_p):
    cnt_p = cnt_p.reindex(range(int(cnt_p.index.min()), int(cnt_p.index.max()) + 1), fill_value=0)   # flat line through years without works
cum_w = pd.DataFrame({'research': cnt_p.research.cumsum(), 'all': (cnt_p.research + cnt_p.other).cumsum()})
fy = UT.basis_year.dropna().astype(int).value_counts().sort_index() if len(UT) else pd.Series(dtype=int)
if len(fy):
    fy = fy.reindex(range(int(fy.index.min()), int(fy.index.max()) + 1), fill_value=0)
cum_t = fy.cumsum()
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
axes[0].plot(cum_w.index, cum_w['all'], color=GREY, lw=2, label='all document types')
axes[0].plot(cum_w.index, cum_w.research, color=SC, lw=2, label='research works')
axes[0].set(xlabel='publication year', ylabel='cumulative works', title=f'Works up to {YEAR_MAX} ({int(papers.has_metadata.sum())} of {len(papers)} with OpenAlex metadata)')
axes[0].legend(loc='upper left')
if len(cum_t):
    axes[1].plot(cum_t.index, cum_t.values, color=TC, lw=2)
else:
    axes[1].text(.5, .5, f'no utility patents up to {YEAR_MAX}', ha='center', va='center', transform=axes[1].transAxes, color=npc.INK2)
axes[1].set(xlabel=f'{PATENT_YEAR_BASIS} year', ylabel='cumulative utility patents', title=f'Patents up to {YEAR_MAX} ({len(UT)} utility patents, by {PATENT_YEAR_BASIS} year)')
fig.suptitle(f'{AUTHOR_NAME} ({AUTHOR_ID}): cumulative output', x=.01, ha='left')
savefig(fig, 'fig1_output'); plt.show()

def fig1_html(drop_empty=False):
    two = len(cum_t) > 0 or not drop_empty
    h = make_subplots(rows=1, cols=2 if two else 1, subplot_titles=[f'Works up to {YEAR_MAX}'] + ([f'Utility patents up to {YEAR_MAX}'] if two else []))
    for col_name, label, color_ in [('all', 'all document types', GREY), ('research', 'research works', SC)]:
        h.add_trace(go.Scatter(x=cum_w.index, y=cum_w[col_name], mode='lines', name=label, line=dict(color=color_, width=2.5),
                               customdata=(cnt_p.research + cnt_p.other if col_name == 'all' else cnt_p.research),
                               hovertemplate=f'{label} %{{x}}<br>cumulative %{{y:,}}<br>that year %{{customdata}}<extra></extra>'), row=1, col=1)
    if two and len(cum_t):
        h.add_trace(go.Scatter(x=cum_t.index, y=cum_t.values, mode='lines', name='utility patents', line=dict(color=TC, width=2.5), customdata=fy.values,
                               hovertemplate='patents %{x}<br>cumulative %{y:,}<br>that year %{customdata}<extra></extra>'), row=1, col=2)
    h.update_yaxes(title_text='cumulative count', col=1, rangemode='tozero'); h.update_yaxes(rangemode='tozero')
    h.update_layout(title=f'{AUTHOR_NAME} ({AUTHOR_ID}): cumulative output', height=450)
    return h
save_html(fig1_html(), 'fig1_output')
dash_only(fig1_html(drop_empty=True), 'fig1_output')

# ---------------- Figure 2: percentile profile of every measure ----------------
SIDES = [('works', R, PAPER_MEASURES, SC), ('patents', UT, PATENT_MEASURES, TC)]
nrow = max(len(PAPER_MEASURES), len(PATENT_MEASURES))
fig, axes = plt.subplots(1, 2, figsize=(16, .5 * nrow + 2), sharex=True)
rng = np.random.default_rng(0)
for ax, (side, frame, measures, col_) in zip(axes, SIDES):
    ylabels = []
    for i, (key, label, pc, cume) in enumerate(measures):
        v = pd.to_numeric(frame.get(pc, pd.Series(dtype=float)), errors='coerce').dropna().to_numpy()
        ylabels.append(f'{label}  (n={len(v)})')
        if len(v):
            ax.scatter(v, i + rng.uniform(-.18, .18, len(v)), s=9, color=col_, alpha=.35, lw=0)
            q1, med, q3 = np.percentile(v, [25, 50, 75])
            ax.plot([q1, q3], [i, i], color=npc.INK, lw=3, solid_capstyle='butt')
            ax.plot([med], [i], marker='|', color=npc.INK, ms=16, mew=2.5)
    ax.set_yticks(range(len(measures)), ylabels); ax.invert_yaxis()
    ax.axvline(.5, color=npc.INK2, lw=.8); ax.axvline(.9, color=npc.INK2, lw=.8, alpha=.5)
    ax.set(xlim=(0, 1), xlabel='cohort percentile (0.5 = typical document of the same cohort)', title=f'{side.capitalize()} (up to {YEAR_MAX})')
    ax.grid(axis='y', visible=False)
fig.suptitle(f'Percentile profile of {AUTHOR_NAME}: each dot is a document; bar = interquartile range, tick = median', x=.01, ha='left')
fig.tight_layout()
savefig(fig, 'fig2_percentile_profile'); plt.show()
def fig2_html(keys=None, drop_empty=False):
    # keys: measures to show (None = all); drop_empty: leave out a side without documents
    sides = [sd for sd in SIDES if not (drop_empty and not len(sd[1]))] or SIDES[:1]   # never zero panels (works panel kept even if empty)
    rows_ = {sd[0]: [m for m in (sd[2] if keys is None else [m for k in keys for m in sd[2] if m[0] == k]) if m[2] in sd[1]] for sd in sides}
    h = make_subplots(rows=1, cols=len(sides), subplot_titles=[f'{sd[0].capitalize()} (up to {YEAR_MAX})' for sd in sides], horizontal_spacing=.28)
    for ci, (side, frame, measures, col_) in enumerate(sides, start=1):
        for key, label, pc, cume in rows_[side]:
            d = frame[frame[pc].notna()]
            raw = RAW_COL[side].get(pc)
            txt = doc_hover(d, side) + f'<br>{label}: ' + d[pc].round(3).astype(str)
            if raw and raw in d:
                txt = txt + ' (value ' + d[raw].round(4).astype('string').fillna('') + ')'
            h.add_trace(go.Box(x=d[pc], y=[label] * len(d), orientation='h', boxpoints='all', jitter=.6, pointpos=0, marker=dict(color=col_, size=5, opacity=.5),
                               line=dict(color=npc.INK, width=1.5), fillcolor='rgba(0,0,0,0)', hovertext=txt, hoverinfo='text', name=label, showlegend=False),
                        row=1, col=ci)
        h.add_vline(x=.5, line_color=npc.INK2, line_width=1, row=1, col=ci)
    h.update_xaxes(range=[0, 1], title_text='cohort percentile')
    h.update_yaxes(autorange='reversed')
    h.update_layout(title=f'Percentile profile of {AUTHOR_NAME} — hover a dot for the document', height=80 + 42 * max([len(v) for v in rows_.values()] + [1]))
    return h
h = fig2_html()
save_html(h, 'fig2_percentile_profile')
dash_only(fig2_html(DASH_MEASURES, drop_empty=True), 'fig2_percentile_profile')

# ---------------- Figure 3: percentiles over the career ----------------
keys = list(dict.fromkeys([m[0] for m in PAPER_MEASURES] + [m[0] for m in PATENT_MEASURES]))
lab_w, lab_t = {m[0]: m[1] for m in PAPER_MEASURES}, {m[0]: m[1] for m in PATENT_MEASURES}
lab = {}
for k in keys:                     # a key shared by works and patents with different meanings gets both labels
    w_, t_ = lab_w.get(k), lab_t.get(k)
    lab[k] = w_ if t_ is None or (w_ and t_.startswith(w_)) else (t_ if w_ is None else f'{w_} | patents: {t_}')
col_of = {'works': {m[0]: m[2] for m in PAPER_MEASURES}, 'patents': {m[0]: m[2] for m in PATENT_MEASURES}}
ncol = 4; nr = math.ceil(len(keys) / ncol)
fig, axes = plt.subplots(nr, ncol, figsize=(16, 3.2 * nr), sharex=True, sharey=True)
for ax, k in zip(axes.flat, keys):
    for side, col_ in [('works', SC), ('patents', TC)]:
        c = col_of[side].get(k)
        if c and f'{side}_{c}' in yearly:
            s = yearly[f'{side}_{c}'].dropna()
            ax.plot(s.index, s.values, color=col_, marker='o', ms=3, lw=1.6, label=side)
    ax.axhline(.5, color=npc.INK2, lw=.8)
    ax.set(ylim=(0, 1), title=lab[k]); ax.title.set_fontsize(9)
for ax in list(axes.flat)[len(keys):]:
    ax.axis('off')
axes.flat[0].legend(loc='lower left', fontsize=8)
fig.supylabel('mean cohort percentile of the year\'s documents'); fig.supxlabel(f'publication year (works) / {PATENT_YEAR_BASIS} year (patents)')
fig.tight_layout()
savefig(fig, 'fig3_percentiles_by_year'); plt.show()
def fig3_html(show_keys):
    nr_ = math.ceil(len(show_keys) / ncol)
    h = make_subplots(rows=nr_, cols=ncol, subplot_titles=[lab[k] for k in show_keys], shared_yaxes=True, vertical_spacing=.09 if nr_ > 1 else .1)
    shown = set()
    for i, k in enumerate(show_keys):
        r_, c_ = i // ncol + 1, i % ncol + 1
        for side, col_ in [('works', SC), ('patents', TC)]:
            c = col_of[side].get(k)
            if c and f'{side}_{c}' in yearly:
                s = yearly[f'{side}_{c}'].dropna()
                if s.empty:
                    continue
                n = yearly.loc[s.index, 'n_works' if side == 'works' else 'n_patents'] if ('n_works' if side == 'works' else 'n_patents') in yearly else s * np.nan
                h.add_trace(go.Scatter(x=s.index, y=s.values, mode='lines+markers', name=side, legendgroup=side, showlegend=side not in shown,
                                       line=dict(color=col_), marker=dict(size=5),
                                       hovertemplate=f'{side} %{{x}}<br>mean percentile %{{y:.3f}}<br>documents %{{customdata}}<extra></extra>', customdata=n),
                            row=r_, col=c_)
                shown.add(side)
        h.add_hline(y=.5, line_color=npc.INK2, line_width=1, row=r_, col=c_)
    h.update_yaxes(range=[0, 1])
    h.update_layout(title=f'{AUTHOR_NAME}: mean cohort percentile per year', height=260 * nr_ + 60)
    return h
h = fig3_html(keys)
save_html(h, 'fig3_percentiles_by_year')
dash_only(fig3_html([k for k in DASH_MEASURES if k in keys]), 'fig3_percentiles_by_year')

# ---------------- Figure 4: how the measures relate (impact vs disruption, novelty, DA) ----------------
PAIRS = [('disruption', 'Disruption CD_5 percentile'), ('novelty', 'Novelty percentile (atypical journal / CPC pairs)'), ('DA', 'Discursive atypicality percentile')]
DASH_PAIRS = [('disruption', 'Disruption (CD, 5 years)'), ('F', 'Foundation share'), ('E', 'Extension share'), ('G', 'Generalization share')]
imp = {'works': 'C_5_pctl', 'patents': 'uniqueC_5_pctl'}
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4), sharey=True)
for j, (k, title) in enumerate(PAIRS):
    ax = axes[j]
    for side, frame, measures, col_ in SIDES:
        c = col_of[side].get(k); yi = imp[side]
        if not c or c not in frame or yi not in frame:
            continue
        d = frame.dropna(subset=[c, yi])
        ax.scatter(d[c], d[yi], s=16, color=col_, alpha=.65, lw=0, marker='o' if side == 'works' else 's', label=f'{side} (n={len(d)})')
    ax.axhline(.5, color=npc.INK2, lw=.8); ax.axvline(.5, color=npc.INK2, lw=.8)
    ax.set(xlim=(0, 1), ylim=(0, 1), xlabel=title, title=title); ax.legend(loc='lower left', fontsize=8)
axes[0].set_ylabel('impact percentile (5-year citations)')
savefig(fig, 'fig4_impact_vs_measures'); plt.show()

def fig4_html(pairs):
    h = make_subplots(rows=1, cols=len(pairs), subplot_titles=[p[1] for p in pairs], shared_yaxes=True)
    shown = set()
    for j, (k, title) in enumerate(pairs):
        for side, frame, measures, col_ in SIDES:
            c = col_of[side].get(k); yi = imp[side]
            if not c or c not in frame or yi not in frame:
                continue
            d = frame.dropna(subset=[c, yi])
            if d.empty:
                continue
            h.add_trace(go.Scatter(x=d[c], y=d[yi], mode='markers', name=side, legendgroup=side, showlegend=side not in shown,
                                   marker=dict(color=col_, size=7, opacity=.7, symbol='circle' if side == 'works' else 'square'),
                                   hovertext=doc_hover(d, side) + f'<br>{title}: ' + d[c].round(3).astype(str) + '<br>impact percentile: ' + d[yi].round(3).astype(str),
                                   hoverinfo='text'), row=1, col=j + 1)
            shown.add(side)
        h.add_hline(y=.5, line_color=npc.INK2, line_width=1, row=1, col=j + 1); h.add_vline(x=.5, line_color=npc.INK2, line_width=1, row=1, col=j + 1)
    h.update_xaxes(range=[0, 1]); h.update_yaxes(range=[0, 1], title_text='impact percentile', col=1)
    h.update_layout(title=f'{AUTHOR_NAME}: impact percentile against ' + ', '.join(p[1].split(' (')[0].lower() for p in pairs), height=500)
    return h
save_html(fig4_html(PAIRS), 'fig4_impact_vs_measures')
dash_only(fig4_html(DASH_PAIRS), 'fig4_impact_vs_measures')


## 8. Citation trajectories (`RUN_TRAJECTORY`)

Yearly citations received by the person's documents up to `YEAR_MAX` (citing years to 2025): paper→paper (`paper_citation_trend.p2p`,
works with metadata), patent→paper (every PCS row of every work, by the citing patent's grant year),
and patent→patent (`uniqueC`: granted and pre-grant-publication citations de-duplicated, with `C` and
`appC` — citations to the patent's pre-grant publication — shown separately). The OpenAlex reference
graph is incomplete for the latest citing years, so every paper→paper count falls after
`COVERAGE_END`; the right panel divides the person's paper→paper citations by all paper→paper
citations of the same citing year (index = 1 in `COVERAGE_END`), which removes that artefact. The
partial year 2026 is not plotted. (`paper_citation_trend.pat2p` comes from an older Reliance-on-Science
release that ends in 2023 and is not used.)

In [ ]:
if RUN_TRAJECTORY:
    t0 = time.time()
    tp = cache('trajectory_works', lambda: con.sql(f'''SELECT cite_year, sum(p2p) AS p2p, count(DISTINCT paper_id) AS n_works_cited
                    FROM read_parquet('{npc.OA / "paper_citation_trend.parquet"}') WHERE paper_id IN (SELECT paper_id FROM ego_p) GROUP BY 1''').df(),
               key=sorted(ego_papers.paper_id), sources=[npc.OA / 'paper_citation_trend.parquet']).set_index('cite_year')
    def traj_patents():
        if not len(util_ids):
            return pd.DataFrame({'cite_year': pd.Series(dtype='int64'), 'C': pd.Series(dtype=float), 'appC': pd.Series(dtype=float),
                                 'uniqueC': pd.Series(dtype=float)})
        a = con.sql(f'''SELECT cite_year, sum(C) AS C, sum(appC) AS appC FROM read_parquet('{npc.PV / "patent_citation_trend.parquet"}')
                        WHERE patent_id IN (SELECT patent_id FROM ego_t) GROUP BY 1''').df()
        b = con.sql(f'''SELECT cite_year, sum(uniqueC) AS uniqueC FROM read_parquet('{npc.PV / "patent_uniqueC_trend.parquet"}')
                        WHERE patent_id IN (SELECT patent_id FROM ego_t) GROUP BY 1''').df()
        return a.merge(b, on='cite_year', how='outer')
    tt = cache('trajectory_patents', traj_patents, key=sorted(util_ids.patent_id),
               sources=[npc.PV / 'patent_citation_trend.parquet', npc.PV / 'patent_uniqueC_trend.parquet']).set_index('cite_year')
    tc = cites_in[cites_in.cite_year.notna()].drop_duplicates(['paper_id', 'invention_id']).groupby('cite_year').size().rename('pcs_all_works')
    pop = npc.population_p2p_by_year(con).set_index('cite_year')
    TIMINGS['trajectory'] = round(time.time() - t0, 1)
    traj = tp.join(tt, how='outer').join(tc, how='outer').join(pop, how='left').sort_index()
    traj = traj[(traj.index >= 1900) & (traj.index <= 2026)]
    for c in ['p2p', 'C', 'appC', 'uniqueC', 'pcs_all_works']:
        if c in traj:
            traj[c] = traj[c].fillna(0)
    traj['p2p_share_index'] = (traj.p2p / traj.p2p_population) / (traj.p2p / traj.p2p_population).get(COVERAGE_END, np.nan)
    save_table(traj.reset_index(), 'citation_trajectory')
    tpl = traj[traj.index <= OBS_END_YEAR]
    fig, axes = plt.subplots(1, 3, figsize=(17, 4))
    axes[0].axvspan(COVERAGE_END + .5, OBS_END_YEAR + .5, color=npc.GRID, alpha=.7, lw=0)   # OpenAlex reference coverage only
    axes[0].plot(tpl.index, tpl.p2p, color=SC, label='paper → paper')
    axes[0].set(xlabel='citing year', ylabel='citations per year', title='Citations to the works (shaded: incomplete reference coverage)')
    axes[0].legend(loc='upper left')
    axes[1].plot(tpl.index, tpl.pcs_all_works, color=LC, label='patent → paper (citing inventions, PCS)')
    if 'uniqueC' in tpl and tpl.uniqueC.sum() > 0:
        axes[1].plot(tpl.index, tpl.uniqueC, color=TC, label='patent → patent (uniqueC)')
        axes[1].plot(tpl.index, tpl.C, color=TC, lw=1, alpha=.6, label='granted → granted (C)')
        axes[1].plot(tpl.index, tpl.appC, color=npc.INK2, lw=1, alpha=.6, label='citations to the pre-grant publication (appC)')
    axes[1].set(xlabel='citing year (patents: grant year)', ylabel='citations per year', title='Patent citations'); axes[1].legend(loc='upper left', fontsize=8)
    d = tpl.p2p_share_index.dropna()
    axes[2].plot(d.index, d.values, color=SC)
    axes[2].axhline(1, color=npc.INK2, lw=.8); axes[2].axvline(COVERAGE_END, color=npc.INK2, lw=.8)
    axes[2].set(xlabel='citing year', ylabel=f'share of all paper→paper citations ({COVERAGE_END} = 1)', title='Coverage-adjusted paper citations')
    if len(d):
        axes[2].set_xlim(max(d.index.min(), d.index.max() - 40), d.index.max() + 1)
    savefig(fig, 'fig5_citation_trajectory'); plt.show()
    h = make_subplots(rows=1, cols=3, subplot_titles=['Citations to the works (shaded: incomplete reference coverage)', 'Patent citations',
                                                      f'Coverage-adjusted paper citations ({COVERAGE_END} = 1)'])
    h.add_trace(go.Scatter(x=tpl.index, y=tpl.p2p, name='paper → paper', line=dict(color=SC)), row=1, col=1)
    h.add_vrect(x0=COVERAGE_END + .5, x1=OBS_END_YEAR + .5, fillcolor=npc.GRID, opacity=.7, line_width=0, row=1, col=1)
    h.add_trace(go.Scatter(x=tpl.index, y=tpl.pcs_all_works, name='patent → paper (citing inventions)', line=dict(color=LC)), row=1, col=2)
    if 'uniqueC' in tpl and tpl.uniqueC.sum() > 0:
        h.add_trace(go.Scatter(x=tpl.index, y=tpl.uniqueC, name='patent → patent (uniqueC)', line=dict(color=TC)), row=1, col=2)
        h.add_trace(go.Scatter(x=tpl.index, y=tpl.C, name='granted → granted (C)', line=dict(color=TC, width=1), opacity=.6), row=1, col=2)
        h.add_trace(go.Scatter(x=tpl.index, y=tpl.appC, name='to the pre-grant publication (appC)', line=dict(color=npc.INK2, width=1), opacity=.6), row=1, col=2)
    d = tpl.p2p_share_index.dropna()
    h.add_trace(go.Scatter(x=d.index, y=d.values, name='share of all paper → paper citations', line=dict(color=SC, dash='solid'), showlegend=False), row=1, col=3)
    h.add_hline(y=1, line_color=npc.INK2, line_width=1, row=1, col=3)
    h.update_layout(title=f'{AUTHOR_NAME}: citations received by the documents up to {YEAR_MAX}', height=460, hovermode='x unified')
    save_html(h, 'fig5_citation_trajectory')
    display(traj.tail(8))
else:
    print('RUN_TRAJECTORY = False: skipped')

## 9. Collaboration networks

**Co-authorship.** Nodes are the person's co-authors; an edge joins two co-authors who appear together
on the person's works, weighted by the number of such works — the ego network induced on the person's
own output, without the person. Works with more than `MAX_TEAM_SIZE` authors are left out of the graph;
`n_shared` counts all shared works, `n_shared_small` those in the graph, and co-authors who share only
consortium works with the person stay in `coauthor_nodes.csv` (`only_large_teams`) but not in the graph,
its statistics or the evolution figure.

**Co-invention.** The same on the person's patents. PatentsView sometimes splits a co-inventor into
several ids; ids with the same normalised first and last name within the person's co-inventors are
merged (`merged_ids`).

**People on both sides.** A co-author and a co-inventor are the same person when (a) the pqrs
crosswalk links them (inventor ids translated to the current release) and their names agree, or (b)
their normalised first + last names are identical (initials-only names: surname + first initial, when
unique on both sides); in both cases middle initials must not contradict each other ('Robert H. Brown'
is not 'Robert A. Brown'). `match_source` records which.

Figures draw the `TOP_K_COLLAB` strongest collaborators of the graph (ties broken by recency, then id);
node size ∝ shared documents, colour = year of first collaboration (grey = undated), edge width ∝
co-occurrences. The layout uses a hidden hub linked to every node, so collaborators connected only
through the person stay near the centre. GraphML files carry every node attribute. Each network is also saved as an interactive page
(`html/fig6..fig8`, the top `HTML_TOP_K` collaborators, hover for name, id, shared documents and years).

In [ ]:
yr_map = papers.set_index('paper_id').year_display.astype('Float64')
co_net = co_rows.merge(ego_papers[['paper_id', 'n_authors_scan']], on='paper_id', how='left')
co_net['year'] = co_net.paper_id.map(yr_map).astype(float)

def ego_network(rows, doc_col, person_col, name_col, year_col, ego_ids, size_col, max_size):
    others = rows[~rows[person_col].isin(ego_ids)]
    nodes = (others.groupby(person_col).agg(n_shared=(doc_col, 'nunique'), first_year=(year_col, 'min'), last_year=(year_col, 'max'),
                                            name=(name_col, lambda s: s.dropna().mode().iloc[0] if s.dropna().size else ''))
             .reset_index().rename(columns={person_col: 'id'}))
    small = others[others[size_col] <= max_size][[doc_col, person_col]].drop_duplicates()
    nodes = nodes.merge(small.groupby(person_col)[doc_col].nunique().rename('n_shared_small'), left_on='id', right_index=True, how='left')
    nodes['n_shared_small'] = nodes.n_shared_small.fillna(0).astype(int)
    nodes['only_large_teams'] = nodes.n_shared_small.eq(0)
    nodes = nodes.sort_values(['n_shared_small', 'n_shared', 'last_year', 'id'], ascending=[False, False, False, True],
                              kind='mergesort', na_position='last').reset_index(drop=True)
    pairs = small.merge(small, on=doc_col)
    pairs = pairs[pairs[f'{person_col}_x'] < pairs[f'{person_col}_y']]
    edges = pairs.groupby([f'{person_col}_x', f'{person_col}_y']).size().reset_index(name='weight')
    edges.columns = ['source', 'target', 'weight']
    G = nx.Graph()
    for r in nodes[~nodes.only_large_teams].itertuples():
        attrs = {'name': str(r.name), 'n_shared': int(r.n_shared_small), 'n_shared_all': int(r.n_shared)}
        if pd.notna(r.first_year):
            attrs['first_year'] = int(r.first_year)
        if pd.notna(r.last_year):
            attrs['last_year'] = int(r.last_year)
        G.add_node(r.id, **attrs)
    G.add_weighted_edges_from(edges.itertuples(index=False, name=None))
    n_excluded = int(rows.loc[rows[size_col] > max_size, doc_col].nunique())
    return nodes, edges, G, n_excluded

def net_stats(G, label):
    keys = ['n_nodes', 'n_edges', 'density', 'avg_clustering', 'n_components', 'largest_component_share', 'isolates']
    if G.number_of_nodes() == 0:
        return {'layer': label, **{k: (0 if k == 'n_nodes' else None) for k in keys}}
    comps = sorted(nx.connected_components(G), key=len, reverse=True)
    return {'layer': label, 'n_nodes': G.number_of_nodes(), 'n_edges': G.number_of_edges(), 'density': round(nx.density(G), 4),
            'avg_clustering': round(nx.average_clustering(G), 3), 'n_components': len(comps),
            'largest_component_share': round(len(comps[0]) / G.number_of_nodes(), 3), 'isolates': nx.number_of_isolates(G)}

coauth_nodes, coauth_edges, G_ca, n_big = ego_network(co_net, 'paper_id', 'author_id_str', 'author_display_name', 'year', set(AUTHOR_IDS),
                                                       'n_authors_scan', MAX_TEAM_SIZE)
print(f'co-authors: {len(coauth_nodes):,} on {co_net.paper_id.nunique():,} works, {int((~coauth_nodes.only_large_teams).sum()):,} of them in the graph '
      f'({n_big} works with more than {MAX_TEAM_SIZE} authors left out; {int(coauth_nodes.only_large_teams.sum()):,} co-authors only on those); '
      f'{len(coauth_edges):,} co-author pairs; {int(coauth_nodes.first_year.isna().sum())} co-authors without a dated shared work')
display(coauth_nodes.head(12))

In [ ]:
# co-invention: rows of the person's patents, co-inventor fragments merged by normalised name
if len(ego_patents):
    ci_rows = con.sql(f'''SELECT i.patent_id, i.inventor_id, i.inventor_sequence, trim(coalesce(i.name_first, '') || ' ' || coalesce(i.name_last, '')) AS inventor_name
                          FROM inv i WHERE i.patent_id IN (SELECT patent_id FROM ego_t_all)
                          QUALIFY row_number() OVER (PARTITION BY i.patent_id, i.inventor_id ORDER BY i.inventor_sequence) = 1''').df()
    py_ = patents.set_index('patent_id')
    yr_t = py_.filing_year.astype('Float64').fillna(pd.to_numeric(py_.grant_year_raw, errors='coerce').astype('Float64'))
    ci_rows['year'] = ci_rows.patent_id.map(yr_t).astype(float)
    others = ci_rows[~ci_rows.inventor_id.isin(inv_sel)]
    keyed = others.drop_duplicates('inventor_id').assign(key=lambda x: x.inventor_name.map(npc.name_key))
    npat = others.groupby('inventor_id').patent_id.nunique()
    keyed['n'] = keyed.inventor_id.map(npat)
    canon = {}
    for k, grp in keyed.dropna(subset=['key']).groupby('key'):
        if len(grp) > 1:
            grp = grp.sort_values(['n', 'inventor_id'], ascending=[False, True])
            main, main_name = grp.inventor_id.iloc[0], grp.inventor_name.iloc[0]
            canon.update({i: main for i, nm_ in zip(grp.inventor_id, grp.inventor_name)
                          if i != main and not npc.middle_conflict(nm_, main_name)})
    ci_rows['inventor_id_raw'] = ci_rows.inventor_id
    ci_rows['inventor_id'] = ci_rows.inventor_id.map(lambda i: canon.get(i, i))
    ci_rows = ci_rows.drop_duplicates(['patent_id', 'inventor_id'])
    ci_rows = ci_rows.merge(ci_rows.groupby('patent_id').inventor_id.nunique().rename('n_inventors_scan'), on='patent_id')
    coinv_nodes, coinv_edges, G_ci, n_big_t = ego_network(ci_rows, 'patent_id', 'inventor_id', 'inventor_name', 'year', set(inv_sel),
                                                          'n_inventors_scan', MAX_TEAM_SIZE)
    merged = pd.Series(canon).groupby(level=0).first()
    coinv_nodes['merged_ids'] = coinv_nodes.id.map(lambda i: ';'.join(sorted(k for k, v in canon.items() if v == i)) or None)
    print(f'co-inventors: {len(coinv_nodes):,} on {ci_rows.patent_id.nunique():,} patents ({len(canon)} duplicate PatentsView ids merged by name); '
          f'{len(coinv_edges):,} co-inventor pairs')
    display(coinv_nodes.head(12))
else:
    ci_rows, canon = pd.DataFrame(columns=['patent_id', 'inventor_id', 'inventor_name', 'year']), {}
    coinv_nodes = pd.DataFrame(columns=['id', 'n_shared', 'first_year', 'last_year', 'name', 'n_shared_small', 'only_large_teams', 'merged_ids'])
    coinv_edges = pd.DataFrame(columns=['source', 'target', 'weight']); G_ci = nx.Graph()
    print('no patents: co-invention network empty')

# people on both sides: (a) pqrs with the release translation and a name guard, (b) identical normalised names
both = []
if len(coauth_nodes) and len(coinv_nodes):
    con.register('ca_ids', coauth_nodes[['id']].rename(columns={'id': 'author_id'}))
    xw = con.sql('SELECT author_id, inventor_id AS pqrs_inventor_id, confidence, n_match FROM pqrs WHERE author_id IN (SELECT author_id FROM ca_ids)').df()
    xw = xw.merge(rmap[rmap.share_of_old >= MIN_RELEASE_SHARE][['id_old', 'id_new']], left_on='pqrs_inventor_id', right_on='id_old', how='left')
    xw['inventor_id'] = xw.id_new.map(lambda i: canon.get(i, i) if isinstance(i, str) else i)
    xw = xw[xw.inventor_id.isin(set(coinv_nodes.id))]
    xw = xw.merge(coauth_nodes[['id', 'name']].rename(columns={'id': 'author_id', 'name': 'author_name'}), on='author_id') \
           .merge(coinv_nodes[['id', 'name']].rename(columns={'id': 'inventor_id', 'name': 'inventor_name'}), on='inventor_id')
    xw['name_agree'] = [npc.names_agree(a, b) and not npc.middle_conflict(a, b) for a, b in zip(xw.author_name, xw.inventor_name)]
    both.append(xw.assign(match_source='pqrs')[['author_id', 'inventor_id', 'author_name', 'inventor_name', 'confidence', 'n_match', 'name_agree', 'match_source']])
    ca_k = coauth_nodes[['id', 'name']].assign(key=lambda x: x.name.map(npc.name_key)).dropna(subset=['key'])
    ci_k = coinv_nodes[['id', 'name']].assign(key=lambda x: x.name.map(npc.name_key)).dropna(subset=['key'])
    nm = ca_k.merge(ci_k, on='key', suffixes=('_a', '_i'))
    # initials-only names ('R.E. Haurwitz'): surname + first initial, when unique on both sides
    def ini_key(n):
        t = npc.name_tokens(n); raw = npc.fold(n).split()
        return (raw[0][0], t[-1]) if t and raw else None
    ca_i = coauth_nodes[['id', 'name']].assign(key=lambda x: x.name.map(ini_key)).dropna(subset=['key'])
    ci_i = coinv_nodes[['id', 'name']].assign(key=lambda x: x.name.map(ini_key)).dropna(subset=['key'])
    ca_i = ca_i[~ca_i.key.duplicated(keep=False)]; ci_i = ci_i[~ci_i.key.duplicated(keep=False)]
    nm = pd.concat([nm, ca_i.merge(ci_i, on='key', suffixes=('_a', '_i'))]).drop_duplicates(['id_a', 'id_i'])
    nm = nm.loc[np.array([not npc.middle_conflict(a, b) for a, b in zip(nm.name_a, nm.name_i)], dtype=bool)]   # 'Robert H. Brown' is not 'Robert A. Brown' (an empty list would select no columns)
    both.append(pd.DataFrame({'author_id': nm.id_a, 'inventor_id': nm.id_i, 'author_name': nm.name_a, 'inventor_name': nm.name_i,
                              'confidence': np.nan, 'n_match': np.nan, 'name_agree': True, 'match_source': 'name'}))
overlap = (pd.concat(both, ignore_index=True) if both else pd.DataFrame(columns=['author_id', 'inventor_id', 'author_name', 'inventor_name', 'confidence',
                                                                                   'n_match', 'name_agree', 'match_source']))
if len(overlap):
    overlap = (overlap.sort_values('match_source', ascending=False)
               .groupby(['author_id', 'inventor_id'], as_index=False)
               .agg(author_name=('author_name', 'first'), inventor_name=('inventor_name', 'first'), confidence=('confidence', 'max'),
                    n_match=('n_match', 'max'), name_agree=('name_agree', 'max'), match_source=('match_source', lambda s: '+'.join(sorted(set(s))))))
    overlap = overlap.merge(coauth_nodes[['id', 'n_shared']].rename(columns={'id': 'author_id', 'n_shared': 'n_shared_works'}), on='author_id') \
                     .merge(coinv_nodes[['id', 'n_shared']].rename(columns={'id': 'inventor_id', 'n_shared': 'n_shared_patents'}), on='inventor_id') \
                     .sort_values(['n_shared_works', 'n_shared_patents'], ascending=False)
ok_ov = overlap[overlap.name_agree == True] if len(overlap) else overlap
both_authors, both_inventors = set(ok_ov.author_id), set(ok_ov.inventor_id)
coauth_nodes['also_coinventor'] = coauth_nodes.id.isin(both_authors)
coinv_nodes['also_coauthor'] = coinv_nodes.id.isin(both_inventors)
print(f'collaborators on both sides: {len(both_authors)} co-authors <-> {len(both_inventors)} co-inventors '
      f'(by source: {ok_ov.match_source.value_counts().to_dict() if len(ok_ov) else {}}); '
      f'{int((overlap.name_agree == False).sum()) if len(overlap) else 0} pqrs links rejected by the name guard (kept in the table)')
display(overlap.head(15))
save_table(coauth_nodes, 'coauthor_nodes'); save_table(coauth_edges, 'coauthor_edges')
save_table(coinv_nodes, 'coinventor_nodes'); save_table(coinv_edges, 'coinventor_edges')
save_table(overlap, 'collaborators_overlap')
stats = pd.DataFrame([net_stats(G_ca, 'co-authorship'), net_stats(G_ci, 'co-invention')])
display(stats); save_table(stats, 'network_stats')

In [ ]:
def hub_layout(H, weight='weight', seed=7, keep_hub=False):
    # spring layout with a hub linked to every node (the person): components connected only through the person stay near
    # the centre; keep_hub returns the person's position too (key '__ego__')
    Hl = H.copy(); hub = '__ego__'
    Hl.add_edges_from((hub, n, {'weight': 1}) for n in H)
    pos = nx.spring_layout(Hl, weight=weight, k=1.4 / math.sqrt(max(H.number_of_nodes(), 2)), seed=seed, iterations=300)
    if not keep_hub:
        pos.pop(hub)
    return pos

def add_ego(fig, H, pos, label):
    # the person as a highlighted node, with faint spokes to every collaborator
    hub = pos['__ego__']; xs, ys = [], []
    for n in H:
        xs += [hub[0], pos[n][0], None]; ys += [hub[1], pos[n][1], None]
    fig.add_trace(go.Scatter(x=xs, y=ys, mode='lines', line=dict(color=npc.INK2, width=.6), opacity=.12, hoverinfo='skip', showlegend=False))
    fig.add_trace(go.Scatter(x=[hub[0]], y=[hub[1]], mode='markers+text', text=[f'<b>{label}</b>'], textposition='bottom center',
                             textfont=dict(size=13, color=npc.INK), marker=dict(symbol='star', size=32, color=npc.INK, line=dict(color='white', width=2)),
                             hovertext=[f'<b>{label}</b><br>the person'], hoverinfo='text', name='the person', showlegend=True))
    fig.update_layout(legend=dict(orientation='h', x=0, xanchor='left', y=-.02, yanchor='top'))   # below the graph, clear of the colour bar

def draw_labels(ax, pos, labels, sizes):
    for n, lab in labels.items():
        x, y = pos[n]
        ax.annotate(lab, (x, y), xytext=(0, 4 + math.sqrt(sizes.get(n, 60)) / 2), textcoords='offset points', ha='center', fontsize=7.5, color=npc.INK)

def ramp(cmap):
    # sequential one-hue ramp without its near-white end, so the earliest collaborators stay visible
    return ListedColormap(plt.get_cmap(cmap)(np.linspace(.3, 1, 256)))

def plotly_scale(cmap):
    cm = ramp(cmap)
    return [[t, to_hex(cm(t))] for t in np.linspace(0, 1, 11)]

def fmt_year(v):
    return str(int(v)) if pd.notna(v) else '?'

def edge_traces(H, pos, color, key='weight', w_max=None):
    # edges binned by weight, one trace per bin (plotly lines cannot vary width within a trace)
    ws = [d.get(key, 1) for _, _, d in H.edges(data=True) if d.get(key, 1) > 0]
    if not ws:
        return []
    w_max = w_max or max(ws); traces = []
    for lo, hi in [(0, .15), (.15, .35), (.35, .65), (.65, 1.01)]:
        xs, ys = [], []
        for a, b, d in H.edges(data=True):
            w = d.get(key, 1)
            if w > 0 and lo <= w / w_max < hi:
                xs += [pos[a][0], pos[b][0], None]; ys += [pos[a][1], pos[b][1], None]
        if xs:
            traces.append(go.Scatter(x=xs, y=ys, mode='lines', line=dict(color=color, width=.6 + 5 * hi), opacity=.25 + .35 * hi,
                                     hoverinfo='skip', showlegend=False))
    return traces

def draw_ego(G, nodes, top_k, title, cmap, fname, highlight=None):
    ids = nodes[nodes.id.isin(set(G.nodes))].head(top_k).id.tolist()
    H = G.subgraph(ids).copy()
    if H.number_of_nodes() == 0:
        print(f'{title}: nothing to draw'); return
    fig, ax = plt.subplots(figsize=(11, 9))
    pos = hub_layout(H)
    w = np.array([d['weight'] for _, _, d in H.edges(data=True)], dtype=float)
    if len(w):
        nx.draw_networkx_edges(H, pos, ax=ax, width=0.4 + 3.0 * w / w.max(), alpha=.3, edge_color=GREY)
    ns = {n: 60 + 900 * H.nodes[n]['n_shared'] / max(H.nodes[m]['n_shared'] for m in H) for n in H}
    dated = [n for n in H if 'first_year' in H.nodes[n]]; undated = [n for n in H if n not in dated]
    rim = lambda L: (['black' if (highlight and n in highlight) else 'white' for n in L], [2.2 if (highlight and n in highlight) else .6 for n in L])
    if undated:
        ec, lw = rim(undated)
        nx.draw_networkx_nodes(H, pos, nodelist=undated, ax=ax, node_size=[ns[n] for n in undated], node_color=GREY, edgecolors=ec, linewidths=lw, alpha=.9)
    if dated:
        fy = np.array([H.nodes[n]['first_year'] for n in dated], dtype=float)
        ec, lw = rim(dated)
        sc = nx.draw_networkx_nodes(H, pos, nodelist=dated, ax=ax, node_size=[ns[n] for n in dated], node_color=fy, cmap=ramp(cmap),
                                    vmin=fy.min(), vmax=max(fy.max(), fy.min() + 1), edgecolors=ec, linewidths=lw, alpha=.92)
        cb = fig.colorbar(sc, ax=ax, fraction=.03, pad=.01); cb.set_label('year of first collaboration')
    draw_labels(ax, pos, {n: H.nodes[n]['name'] for n in ids[:LABEL_TOP] if n in H}, ns)
    ax.set_title(f'{title}\nnode size ∝ shared documents with {AUTHOR_NAME}; edge width ∝ documents shared by the pair; grey = undated'
                 + ('; black rim = also on the other side' if highlight else ''), fontsize=10)
    ax.axis('off'); ax.grid(False)
    savefig(fig, fname); plt.show()

def network_html(G, nodes, top_k, title, cmap, fname, highlight=None, ego=None, highlight_label='also on the other side', label_ids=True):
    # interactive version: more collaborators, pan / zoom, hover for every person
    ids = nodes[nodes.id.isin(set(G.nodes))].head(top_k).id.tolist()
    H = G.subgraph(ids).copy()
    if H.number_of_nodes() == 0:
        return
    pos = hub_layout(H, keep_hub=ego is not None)
    info = nodes.set_index('id')
    fig = go.Figure(edge_traces(H, pos, GREY))
    if ego is not None:
        add_ego(fig, H, pos, ego)
    nmax = max(H.nodes[n]['n_shared'] for n in H)
    size = {n: 7 + 30 * math.sqrt(H.nodes[n]['n_shared'] / nmax) for n in H}
    lab = set(ids[:LABEL_TOP]) | ({n for n in H if n in highlight} if highlight else set())   # highlighted nodes always carry a label
    def hov(n):
        r = info.loc[n]
        return (f"<b>{r['name']} ({n})</b><br>shared documents: {int(r.n_shared)} (in the graph {int(r.n_shared_small)})"
                f"<br>collaboration: {fmt_year(r.first_year)}–{fmt_year(r.last_year)}" + (f'<br><b>{highlight_label}</b>' if highlight and n in highlight else ''))
    dated = [n for n in H if 'first_year' in H.nodes[n]]; undated = [n for n in H if n not in dated]
    for group, is_dated in [(undated, False), (dated, True)]:
        if not group:
            continue
        marker = dict(size=[size[n] for n in group],
                      line=dict(color=[npc.INK if highlight and n in highlight else 'white' for n in group],
                                width=[3.5 if highlight and n in highlight else 1 for n in group]))
        if is_dated:
            marker.update(color=[H.nodes[n]['first_year'] for n in group], colorscale=plotly_scale(cmap), showscale=True,
                          colorbar=dict(title='first<br>collaboration'))
        else:
            marker.update(color=GREY)
        fig.add_trace(go.Scatter(x=[pos[n][0] for n in group], y=[pos[n][1] for n in group], mode='markers+text',
                                 text=[(f"{info.loc[n, 'name']} ({n})" if label_ids else info.loc[n, 'name']) if n in lab else '' for n in group],
                                 textposition='top center', textfont=dict(size=10),
                                 hovertext=[hov(n) for n in group], hoverinfo='text', marker=marker, showlegend=False))
    fig.update_xaxes(visible=False); fig.update_yaxes(visible=False, scaleanchor='x')
    fig.update_layout(title=f'{title}<br><sup>top {len(ids)} collaborators; size ∝ shared documents; colour = first collaboration (grey = undated)'
                            + (f'; black rim = {highlight_label}' if highlight else '') + ('; star = the person' if ego else '') + '; scroll to zoom, hover for details</sup>',
                      height=900, dragmode='pan', plot_bgcolor='white')
    save_html(fig, fname)

draw_ego(G_ca, coauth_nodes, TOP_K_COLLAB, f'Co-authorship ego network — top {TOP_K_COLLAB} co-authors', 'Blues', 'fig6_coauthorship_network', both_authors)
network_html(G_ca, coauth_nodes, HTML_TOP_K, f'Co-authorship ego network of {AUTHOR_NAME} (works up to {YEAR_MAX})', 'Blues', 'fig6_coauthorship_network', both_authors,
             ego=f'{AUTHOR_NAME} ({AUTHOR_ID})')
if G_ci.number_of_nodes():
    draw_ego(G_ci, coinv_nodes, TOP_K_COLLAB, f'Co-invention ego network — top {TOP_K_COLLAB} co-inventors', 'Oranges', 'fig7_coinvention_network', both_inventors)
    network_html(G_ci, coinv_nodes, HTML_TOP_K, f'Co-invention ego network of {AUTHOR_NAME} (patents up to {YEAR_MAX})', 'Oranges', 'fig7_coinvention_network', both_inventors,
                 ego=f"{INVENTOR_NAME or AUTHOR_NAME} ({', '.join(inv_sel)})")

# combined network: the union of the co-authorship and the co-invention network; co-inventors that are co-authors merged onto
# the author node. The PNG keeps the top TOP_K_COLLAB of each side, the interactive page the same HTML_TOP_K per side as the
# separate interactive networks, so it holds every person of both of them.
def combined_graph(k, keep_hub=False):
    top_a = set(coauth_nodes[~coauth_nodes.only_large_teams].head(k).id)
    top_i = set(coinv_nodes[~coinv_nodes.only_large_teams].head(k).id)
    U = nx.Graph()
    for nid in top_a | {inv2auth.get(i, i) for i in top_i}:
        if nid in ca_ns.index:
            U.add_node(nid, name=ca_nm[nid], papers=int(ca_ns[nid]), patents=int(pat_by_auth.get(nid, 0)))
        else:
            U.add_node(nid, name=ci_nm[nid], papers=0, patents=int(ci_ns[nid]))
    for r in coauth_edges[coauth_edges.source.isin(U) & coauth_edges.target.isin(U)].itertuples():
        U.add_edge(r.source, r.target, w_paper=int(r.weight), w_patent=0)
    for r in coinv_edges.itertuples():
        a, b = inv2auth.get(r.source, r.source), inv2auth.get(r.target, r.target)
        if a == b or a not in U or b not in U:
            continue
        if U.has_edge(a, b):
            U.edges[a, b]['w_patent'] += int(r.weight)
        else:
            U.add_edge(a, b, w_paper=0, w_patent=int(r.weight))
    pos = hub_layout(U, weight=None, keep_hub=keep_hub)
    ep = [(a, b) for a, b, d in U.edges(data=True) if d['w_paper'] > 0]
    et = [(a, b) for a, b, d in U.edges(data=True) if d['w_patent'] > 0]
    w_max = max([U.edges[e]['w_paper'] for e in ep] + [U.edges[e]['w_patent'] for e in et] + [1])
    role = {n: 'both' if (d['papers'] and d['patents']) else ('paper' if d['papers'] else 'patent') for n, d in U.nodes(data=True)}
    tot = {n: d['papers'] + d['patents'] for n, d in U.nodes(data=True)}
    top_lab = sorted(U, key=lambda n: -tot[n])[:max(LABEL_TOP, 16)]
    return U, pos, ep, et, w_max, role, tot, top_lab

if G_ci.number_of_nodes() and G_ca.number_of_nodes():
    inv2auth = (ok_ov.sort_values('n_shared_works', ascending=False).drop_duplicates('inventor_id').set_index('inventor_id').author_id.to_dict()
                if len(ok_ov) and 'n_shared_works' in ok_ov else {})     # nobody on both sides: nothing to merge
    ca_ns = coauth_nodes.set_index('id').n_shared_small; ca_nm = coauth_nodes.set_index('id').name
    ci_ns = coinv_nodes.set_index('id').n_shared_small; ci_nm = coinv_nodes.set_index('id').name
    pat_by_auth = (pd.Series({i: ci_ns.get(i, 0) for i in inv2auth}).groupby(pd.Series(inv2auth)).sum() if inv2auth
                   else pd.Series(dtype='int64'))
    col = {'paper': SC, 'patent': TC, 'both': LC}

    U, pos, ep, et, w_max, role, tot, top_lab = combined_graph(TOP_K_COLLAB)
    fig, ax = plt.subplots(figsize=(12, 10))
    if ep:
        nx.draw_networkx_edges(U, pos, edgelist=ep, ax=ax, edge_color=SC, alpha=.3, width=[0.4 + 3 * U.edges[e]['w_paper'] / w_max for e in ep])
    if et:
        nx.draw_networkx_edges(U, pos, edgelist=et, ax=ax, edge_color=TC, alpha=.55, width=[0.4 + 3 * U.edges[e]['w_patent'] / w_max for e in et])
    ns = {n: 60 + 900 * tot[n] / max(tot.values()) for n in U}
    nx.draw_networkx_nodes(U, pos, ax=ax, node_color=[col[role[n]] for n in U], node_size=[ns[n] for n in U], alpha=.9, edgecolors='white', linewidths=.8)
    draw_labels(ax, pos, {n: U.nodes[n]['name'] for n in top_lab}, ns)
    n_both = sum(1 for r in role.values() if r == 'both')
    ax.legend(handles=[Line2D([], [], marker='o', color='w', markerfacecolor=col['paper'], markersize=10, label='co-author only'),
                       Line2D([], [], marker='o', color='w', markerfacecolor=col['patent'], markersize=10, label='co-inventor only'),
                       Line2D([], [], marker='o', color='w', markerfacecolor=col['both'], markersize=10, label=f'both ({n_both})'),
                       Line2D([], [], color=SC, lw=2, label='co-authorship edge'), Line2D([], [], color=TC, lw=2, label='co-invention edge')],
              loc='upper left', bbox_to_anchor=(1.0, 1.0), fontsize=9)
    ax.set_title(f'Combined collaboration network of {AUTHOR_NAME}: top {TOP_K_COLLAB} co-authors and top {TOP_K_COLLAB} co-inventors\n'
                 f'node size ∝ shared works + shared patents; edge width ∝ shared documents on one scale (max {w_max})', fontsize=10)
    ax.axis('off'); ax.grid(False)
    savefig(fig, 'fig8_combined_network'); plt.show()

    U, pos, ep, et, w_max, role, tot, top_lab = combined_graph(HTML_TOP_K, keep_hub=True)
    hf = go.Figure()
    for key_, color_ in [('w_paper', SC), ('w_patent', TC)]:
        for tr in edge_traces(U, pos, color_, key=key_, w_max=w_max):
            hf.add_trace(tr)
    add_ego(hf, U, pos, f'{AUTHOR_NAME} ({AUTHOR_ID})')
    for rl, name_ in [('paper', 'co-author only'), ('patent', 'co-inventor only'), ('both', 'both')]:
        grp = [n for n in U if role[n] == rl]
        if not grp:
            continue
        hf.add_trace(go.Scatter(x=[pos[n][0] for n in grp], y=[pos[n][1] for n in grp], mode='markers+text', name=f'{name_} ({len(grp)})',
                                text=[f"{U.nodes[n]['name']} ({n})" if n in top_lab else '' for n in grp], textposition='top center', textfont=dict(size=10),
                                marker=dict(color=col[rl], size=[7 + 30 * math.sqrt(tot[n] / max(tot.values())) for n in grp], line=dict(color='white', width=1)),
                                hovertext=[f"<b>{U.nodes[n]['name']} ({n})</b><br>shared works: {U.nodes[n]['papers']}<br>shared patents: {U.nodes[n]['patents']}" for n in grp],
                                hoverinfo='text'))
    hf.update_xaxes(visible=False); hf.update_yaxes(visible=False, scaleanchor='x')
    hf.update_layout(title=f'Combined collaboration network of {AUTHOR_NAME} (up to {YEAR_MAX}): top {HTML_TOP_K} co-authors and top {HTML_TOP_K} co-inventors'
                           f'<br><sup>{U.number_of_nodes()} people; blue edges: co-authorship, orange edges: co-invention; size ∝ shared works + patents; '
                           f'scroll to zoom, hover for details</sup>', height=900, dragmode='pan', plot_bgcolor='white')
    save_html(hf, 'fig8_combined_network')

# evolution of the collaborator pool (graph members only; undated collaborators counted in the titles)
def collab_by_year(nodes):
    fy = nodes.first_year.dropna().astype(int)
    new = fy.value_counts().sort_index()
    return pd.DataFrame({'new': new, 'cumulative': new.cumsum()})
ca_g = coauth_nodes[~coauth_nodes.only_large_teams]; ci_g = coinv_nodes[~coinv_nodes.only_large_teams.astype(bool)] if len(coinv_nodes) else coinv_nodes
ev_a, ev_i = collab_by_year(ca_g), collab_by_year(ci_g)
panels = [(ev_a, SC, 'co-authors', ca_g), (ev_i, TC, 'co-inventors', ci_g)]
fig, axes = plt.subplots(2, 2, figsize=(14, 6.5), sharex='col')
for j, (ev, col_, lab_, nodes_) in enumerate(panels):
    if len(ev):
        axes[0, j].bar(ev.index, ev.new, color=col_, width=.8)
        axes[1, j].plot(ev.index, ev.cumulative, color=col_)
    axes[0, j].set(ylabel=f'new {lab_}', title=f'{lab_.capitalize()} in the graph: {len(nodes_)} ({int(nodes_.first_year.isna().sum())} undated not shown)')
    axes[1, j].set(xlabel='year of first collaboration', ylabel=f'cumulative {lab_}')
savefig(fig, 'fig9_collaborator_evolution'); plt.show()
h = make_subplots(rows=2, cols=2, shared_xaxes='columns', subplot_titles=['new co-authors per year', 'new co-inventors per year',
                                                                          'cumulative co-authors', 'cumulative co-inventors'])
for j, (ev, col_, lab_, nodes_) in enumerate(panels, start=1):
    if len(ev):
        h.add_bar(x=ev.index, y=ev.new, marker_color=col_, name=f'new {lab_}', row=1, col=j)
        h.add_trace(go.Scatter(x=ev.index, y=ev.cumulative, line=dict(color=col_), name=f'cumulative {lab_}'), row=2, col=j)
h.update_layout(title=f'{AUTHOR_NAME}: collaborators by year of first collaboration (up to {YEAR_MAX})', height=650, showlegend=False)
save_html(h, 'fig9_collaborator_evolution')
ev = ev_a.add_prefix('coauthors_').join(ev_i.add_prefix('coinventors_'), how='outer').sort_index()
save_table(ev.reset_index().rename(columns={'index': 'year'}), 'collaborators_by_year')

for G, nodes, flag, fname in [(G_ca, coauth_nodes, 'also_coinventor', 'coauthorship_network.graphml'), (G_ci, coinv_nodes, 'also_coauthor', 'coinvention_network.graphml')]:
    if G.number_of_nodes():
        Hg = G.copy()
        nx.set_node_attributes(Hg, {r.id: int(bool(getattr(r, flag))) for r in nodes.itertuples()}, 'also_other_side')
        nx.write_graphml(Hg, OUT / fname); WRITTEN.append(OUT / fname)
        print('graph  ->', (OUT / fname).relative_to(npc.NP))

## 9b. Networks of countries, institutions and assignees

The same ego-network construction with organisations as nodes: **works × countries** (countries of the
authors' institutions, `paper_author_country`), **works × institutions** (every author's OpenAlex institution
on works with at most `MAX_TEAM_SIZE` authors), **patents × inventor countries** (`patent_inventor_country`)
and **patents × assignees** (PatentsView disambiguated assignees). Node size ∝ the person's documents with
that country or organisation, edges join two of them on the same document, colour = first year. The
person's own institutions and country are nodes too, usually the hubs.

In [ ]:
def entity_graph(rows, doc_col, ent_col, name_map, year_map):
    # nodes = entities on the person's documents (n_shared = documents with the entity), edges = entities on the same document
    rows = rows.dropna(subset=[ent_col]).drop_duplicates([doc_col, ent_col]).copy()
    rows['year'] = rows[doc_col].map(year_map).astype(float)
    nodes = (rows.groupby(ent_col).agg(n_shared=(doc_col, 'nunique'), first_year=('year', 'min'), last_year=('year', 'max'))
             .reset_index().rename(columns={ent_col: 'id'}))
    nodes['name'] = nodes.id.map(name_map).fillna(nodes.id).astype(str)
    nodes['n_shared_small'] = nodes.n_shared
    nodes['only_large_teams'] = False
    nodes = nodes.sort_values(['n_shared', 'last_year', 'id'], ascending=[False, False, True], kind='mergesort', na_position='last').reset_index(drop=True)
    pr = rows[[doc_col, ent_col]]
    pairs = pr.merge(pr, on=doc_col)
    pairs = pairs[pairs[f'{ent_col}_x'] < pairs[f'{ent_col}_y']]
    edges = pairs.groupby([f'{ent_col}_x', f'{ent_col}_y']).size().reset_index(name='weight')
    edges.columns = ['source', 'target', 'weight']
    G = nx.Graph()
    for r in nodes.itertuples():
        a = {'name': r.name, 'n_shared': int(r.n_shared)}
        if pd.notna(r.first_year):
            a['first_year'] = int(r.first_year)
        if pd.notna(r.last_year):
            a['last_year'] = int(r.last_year)
        G.add_node(r.id, **a)
    G.add_weighted_edges_from(edges.itertuples(index=False, name=None))
    return nodes, edges, G

def split_rows(frame, doc_col, list_col, out_col):
    s = frame[[doc_col, list_col]].dropna()
    s = s.assign(**{out_col: s[list_col].astype(str).str.split(';')}).explode(out_col)
    s[out_col] = s[out_col].str.strip()
    return s[s[out_col] != ''][[doc_col, out_col]]

ENTITY = {}
wyear = papers.set_index('paper_id').year_display.astype('Float64')
# works x countries of the authors' institutions (paper_author_country), works x institutions (all authors, consortia excluded)
ENTITY['works_country'] = entity_graph(split_rows(papers, 'paper_id', 'countries', 'country'), 'paper_id', 'country', {}, wyear)
inst_names = {}
if INST_PQ.exists():
    it = pd.read_parquet(INST_PQ).drop_duplicates('institution_id')
    inst_names = (it.display_name.fillna(it.institution_id) + ' (' + it.country_code.fillna('?') + ')').set_axis(it.institution_id).to_dict()
small_works = set(ego_papers.loc[ego_papers.n_authors_scan <= MAX_TEAM_SIZE, 'paper_id'])
ENTITY['works_institution'] = entity_graph(split_rows(co_rows[co_rows.paper_id.isin(small_works)], 'paper_id', 'institution_ids', 'institution'),
                                           'paper_id', 'institution', inst_names, wyear)
# patents x inventor countries, patents x assignees
pyear = patents.set_index('patent_id').basis_year.astype('Float64') if len(patents) else pd.Series(dtype=float)
ENTITY['patents_country'] = entity_graph(split_rows(patents, 'patent_id', 'countries', 'country'), 'patent_id', 'country', {}, pyear) \
    if len(patents) and 'countries' in patents else None
if len(patents) and 'assignee_list' in patents:
    arows = split_rows(patents, 'patent_id', 'assignee_list', 'assignee')
    con.register('asg_ids', pd.DataFrame({'assignee_id': arows.assignee.astype('string').unique()}))
    anames = con.sql(f'''SELECT assignee_id, mode(coalesce(nullif(disambig_assignee_organization, ''),
                                                           trim(coalesce(disambig_assignee_individual_name_first, '') || ' ' ||
                                                                coalesce(disambig_assignee_individual_name_last, '')))) AS name
                         FROM read_parquet('{npc.CACHE / 'g_assignee_disambiguated.parquet'}')
                         WHERE assignee_id IN (SELECT assignee_id FROM asg_ids) GROUP BY 1''').df().set_index('assignee_id').name.to_dict()
    ENTITY['patents_assignee'] = entity_graph(arows, 'patent_id', 'assignee', anames, pyear)
else:
    ENTITY['patents_assignee'] = None

ENTITY_SPEC = {
    'works_country': ('Countries on the works', 'Blues', 'fig11_works_country_network'),
    'works_institution': (f'Institutions on the works (works with at most {MAX_TEAM_SIZE} authors)', 'Blues', 'fig12_works_institution_network'),
    'patents_country': ('Inventor countries on the patents', 'Oranges', 'fig13_patents_country_network'),
    'patents_assignee': ('Assignees of the patents', 'Oranges', 'fig14_patents_assignee_network'),
}
# the person's own affiliations, highlighted in the interactive pages: institutions of the person's authorships; assignees whose name
# shares a distinctive word with those institutions (DeepMind Technologies Limited ~ DeepMind (United Kingdom)) or is the person
own_inst = set(ego_rows.institution_id.dropna())
aff_tok = set().union(*[npc.org_tokens(n) for n in aff.display_name.dropna()]) if 'aff' in globals() and len(aff) else set()
OWN = {'works_institution': own_inst,
       'works_country': set(aff.country_code.dropna()) if 'aff' in globals() and len(aff) else set()}
if ENTITY.get('patents_assignee') is not None:
    an = ENTITY['patents_assignee'][0]
    OWN['patents_assignee'] = set(an.loc[an.name.map(lambda s: bool(npc.org_tokens(s) & aff_tok)
                                                     or (npc.name_key(s) is not None and npc.name_key(s) == npc.name_key(AUTHOR_NAME))).astype(bool), 'id'])
print('highlighted own affiliations:', {k: len(v) for k, v in OWN.items()})
ent_stats = []
for key, res in ENTITY.items():
    if res is None or not res[2].number_of_nodes():
        print(f'{key}: no data'); continue
    nodes_, edges_, G_ = res
    title, cmap, fname = ENTITY_SPEC[key]
    print(f'{key}: {len(nodes_)} nodes, {len(edges_)} edges; top: ' + ', '.join(f'{n} ({c})' for n, c in zip(nodes_.name.head(6), nodes_.n_shared.head(6))))
    save_table(nodes_.drop(columns=['n_shared_small', 'only_large_teams']), f'{key}_nodes'); save_table(edges_, f'{key}_edges')
    draw_ego(G_, nodes_, TOP_K_COLLAB, f'{title} — {AUTHOR_NAME} (up to {YEAR_MAX})', cmap, fname)
    network_html(G_, nodes_, HTML_TOP_K, f'{title} — {AUTHOR_NAME} (up to {YEAR_MAX})', cmap, fname,
                 highlight=OWN.get(key), highlight_label="the person's affiliation", label_ids=False)   # places and organisations: names only
    ent_stats.append(net_stats(G_, key))
stats = pd.concat([stats, pd.DataFrame(ent_stats)], ignore_index=True) if ent_stats else stats
save_table(stats, 'network_stats')

## 10. Dashboard

The profile in one page, centred on citations, disruption, Foundation / Extension / Generalization and citations from
books (`DASH_MEASURES`; novelty, conventionality, DA and sleeping beauty stay in the tables and figures above): the headline
numbers and, on one scrolling page with section links at the top, the percentile summary, every document against its cohort, output, mean percentile per year,
citations received, book citations, the co-authorship, co-invention and combined networks, new collaborators per year and the
top works and patents; people appear as name (id) (overview, over time, measures, books, people, countries and organisations, documents).
It is saved as `Nobel Prize/output/dashboard/<prize year>_<field>_<name>_<author id>.html`, a standalone HTML
file (plotly is inside the page, so it opens on its own, offline) that follows the light / dark theme of the
browser. Prize year and field come from the PrizeAtlas crawl (`Data/prizeatlas/`, matched on the OpenAlex id,
then ORCID, then a unique name) or the laureate author bridge; `NP_PRIZE_YEAR` / `NP_PRIZE_FIELD` set them by
hand, and they are `NA` for an author who is not a laureate. Several prizes are joined with `-`
(e.g. `1956-1972_Physics_John-Bardeen_A5110170702.html`).

In [ ]:
import html as _html
import plotly

def esc(x):
    if x is None or (isinstance(x, float) and not math.isfinite(x)) or x is pd.NA:
        return ''
    return _html.escape(str(x))

def pbar(v):
    # a percentile as a small bar plus its value
    if v is None or pd.isna(v):
        return '<td class="num muted">–</td>'
    return f'<td class="num"><span class="pbar"><i style="width:{100 * float(v):.0f}%"></i></span>{float(v):.2f}</td>'

def fig_json(name):
    # the card title names the figure on the page, so the figure's own title is dropped (subplot titles stay)
    if name not in DASH:
        return None
    f = go.Figure(DASH[name])
    f.update_layout(title=None, margin=dict(t=50 if f.layout.annotations else 20))
    npc.clip_years(f, YEAR_MAX)                   # calendar-year axes end at YEAR_MAX (later years are incomplete)
    for a in f.layout.annotations or []:          # the shaded incomplete years are cut away, so is their note
        if a.text and '(shaded:' in a.text:
            a.text = a.text.split(' (shaded:')[0]
    return f.to_json().replace('</', '<\\/')

# ---------- tables ----------
def psum_table():
    head = '<tr><th>Measure</th><th class="num">Works: median</th><th class="num">top 10 %</th><th class="num">top 1 %</th><th class="num">n</th>' \
           '<th class="num">Patents: median</th><th class="num">top 10 %</th><th class="num">top 1 %</th><th class="num">n</th></tr>'
    rows = []
    keys = [k for k in DASH_MEASURES if k in set(PSUM.key)]      # the dashboard's measures (every measure stays in percentile_summary.csv)
    for k in keys:
        w = PSUM[(PSUM.side == 'works') & (PSUM.key == k)]
        t = PSUM[(PSUM.side == 'patents') & (PSUM.key == k)]
        lab = (w.measure.iloc[0] if len(w) else t.measure.iloc[0])
        cells = [f'<td>{esc(lab)}</td>']
        for part in (w, t):
            if len(part) and part.n.iloc[0] > 0:
                r = part.iloc[0]
                cells += [pbar(r.median_pctl), f'<td class="num">{r.share_top10:.0%}</td>', f'<td class="num">{r.share_top1:.0%}</td>', f'<td class="num">{int(r.n)}</td>']
            else:
                cells += ['<td class="num muted">–</td>'] * 4
        rows.append('<tr>' + ''.join(cells) + '</tr>')
    return f'<div class="tablewrap"><table>{head}{"".join(rows)}</table></div>'

def works_table(n=25):
    cols = ['C_5_pctl', 'CD_5_rpctl_mid', 'F_5_pctl', 'E_5_pctl', 'G_5_pctl', 'book_cites_pctl']
    d = R.sort_values(['C_5_pctl', 'C_all'], ascending=False).head(n)
    head = '<tr><th>Work</th><th class="num">Year</th><th>Venue</th><th class="num">Impact</th><th class="num">Disruption</th><th class="num">Foundation</th>' \
           '<th class="num">Extension</th><th class="num">Generalization</th>' \
           '<th class="num">Books</th><th class="num">Citing books</th><th class="num">Citing patents</th></tr>'
    rows = []
    for r in d.itertuples():
        title = getattr(r, 'title', None) if 'title' in d else None
        label = esc(title) if isinstance(title, str) and title else esc(r.paper_id)
        rows.append(f'<tr><td><a href="https://openalex.org/{esc(r.paper_id)}" target="_blank" rel="noopener">{label}</a>'
                    f'<div class="id">{esc(r.paper_id)}</div></td><td class="num">{esc(r.year)}</td><td>{esc(r.journal)}</td>'
                    + ''.join(pbar(getattr(r, c)) for c in cols)
                    + f'<td class="num">{int(r.n_book_cites) if pd.notna(r.n_book_cites) else "–"}</td><td class="num">{int(r.pcs_C_total)}</td></tr>')
    return f'<div class="tablewrap"><table>{head}{"".join(rows)}</table></div>'

def patents_table(n=20):
    if not len(UT):
        return '<p class="muted">No patents up to the cut-off year.</p>'
    d = UT.sort_values('uniqueC_5_pctl', ascending=False).head(n)
    head = '<tr><th>Patent</th><th class="num">Grant</th><th class="num">Impact</th><th class="num">Disruption</th><th class="num">Foundation</th>' \
           '<th class="num">Extension</th><th class="num">Generalization</th><th class="num">Papers cited</th></tr>'
    rows = [f'<tr><td>{esc(r.patent_title)}<div class="id">US {esc(r.patent_id)}</div></td><td class="num">{esc(r.grant_year)}</td>'
            + ''.join(pbar(getattr(r, c, None)) for c in ['uniqueC_5_pctl', 'CD_5_rpctl_mid', 'F_5_pctl', 'E_5_pctl', 'G_5_pctl'])
            + f'<td class="num">{int(r.n_paper_refs)}</td></tr>' for r in d.itertuples()]
    return f'<div class="tablewrap"><table>{head}{"".join(rows)}</table></div>'

def books_table(n=12):
    if not len(books):
        return '<p class="muted">No citing books found.</p>'
    src_ = books.drop_duplicates('book_id').book_source.replace('', np.nan).dropna().value_counts().head(n)   # books without a source record are left out
    rows = ''.join(f'<tr><td>{esc(k)}</td><td class="num">{v}</td></tr>' for k, v in src_.items())
    return f'<div class="tablewrap"><table><tr><th>Source of the citing books</th><th class="num">Books</th></tr>{rows}</table></div>'

# ---------- page ----------
NOBEL = ({} if NOBEL_LOOKUP == 'off' else
         npc.nobel_identity(list(dict.fromkeys(AUTHOR_IDS + INPUT_AUTHOR_IDS)), AUTHOR_NAME, orcids if 'orcids' in globals() else [],
                            use_name=NOBEL_LOOKUP == 'auto'))
if PRIZE_YEAR or PRIZE_FIELD:                    # explicit parameters win over the lookup
    same = (PRIZE_YEAR or NOBEL.get('year'), PRIZE_FIELD or NOBEL.get('field')) == (NOBEL.get('year'), NOBEL.get('field'))
    NOBEL = {**NOBEL, 'year': PRIZE_YEAR or NOBEL.get('year'), 'field': PRIZE_FIELD or NOBEL.get('field'),
             'rule': NOBEL.get('rule') if same and NOBEL.get('rule') else 'parameters'}
nobel_line = f"Nobel Prize · {', '.join(NOBEL['prizes'])} · " if NOBEL.get('prizes') else ''
PLOTLY_TAG = npc.plotly_tag(standalone=True)      # plotly.js inside the page: one file that works offline
top_aff = ', '.join(aff.display_name.dropna().head(3)) if 'aff' in globals() and len(aff) else ''
w_imp = PSUM[(PSUM.side == 'works') & (PSUM.key == 'impact5')].iloc[0]

# ---------- patent -> paper citations (Reliance on Science) ----------
N_CITING = int(cites_in.invention_id.nunique())
N_CITING_SELF = int(cites_in.loc[cites_in.self_citation, 'invention_id'].nunique())
first_cite = cites_in.groupby('paper_id').cite_year.min()
def short_title(r, k=60):
    t = getattr(r, 'title', None) if 'title' in papers else None
    return (t[:k] + ('…' if len(t) > k else '')) if isinstance(t, str) and t else r.paper_id
if N_CITING:
    inv_ = cites_in.dropna(subset=['cite_year']).groupby('invention_id').agg(cite_year=('cite_year', 'min'), own=('self_citation', 'max'))
    py_ = inv_.groupby(['cite_year', 'own']).size().unstack(fill_value=0)
    top_ = papers[papers.n_citing_inventions > 0].sort_values(['n_citing_inventions', 'C_all'], ascending=False).head(15).iloc[::-1]
    hp = make_subplots(rows=1, cols=2, column_widths=[.45, .55], horizontal_spacing=.32,
                       subplot_titles=['Citing inventions per year (year of the citing patent)', 'Works most cited by patents'])
    for own, name_, color_ in [(False, 'other inventors', TC), (True, "the person's own patents", LC)]:
        if own in py_:
            hp.add_bar(x=py_.index, y=py_[own], name=name_, marker_color=color_, row=1, col=1,
                       hovertemplate=f'{name_} %{{x}}<br>citing inventions %{{y}}<extra></extra>')
    hp.add_trace(go.Bar(x=top_.n_citing_inventions, y=[short_title(r) for r in top_.itertuples()], orientation='h', marker_color=TC, showlegend=False,
                        hovertext=[f"{short_title(r, 200)}<br>{r.paper_id} · {r.year_display}<br>citing inventions: {r.n_citing_inventions} "
                                   f"(own {r.n_citing_self})<br>first cited by a patent: {first_cite.get(r.paper_id, '–')}" for r in top_.itertuples()],
                        hoverinfo='text'), row=1, col=2)
    hp.update_xaxes(title_text='citing year', row=1, col=1); hp.update_yaxes(title_text='distinct inventions', row=1, col=1)
    hp.update_xaxes(title_text='distinct citing inventions', row=1, col=2)
    hp.update_layout(barmode='stack', height=560, legend=dict(orientation='h', y=-.18))
    dash_only(hp, 'figP_patent_citations')
science_link = (f"The person's {refs_out.patent_id.nunique()} patents cite {refs_out.cited_paper_id.nunique()} papers, "
                f"{refs_out.loc[refs_out.self_reference, 'cited_paper_id'].nunique()} of them the person's own works."
                if 'refs_out' in globals() and len(refs_out) else '')

def patent_cited_table(n=20):
    d = papers[papers.n_citing_inventions > 0].sort_values(['n_citing_inventions', 'C_all'], ascending=False).head(n)
    if d.empty:
        return '<p class="muted">No patent cites the works.</p>'
    head = ('<tr><th>Work</th><th class="num">Year</th><th class="num">Citing inventions</th><th class="num">Own patents</th>'
            '<th class="num">First patent citation</th><th class="num">Cited by patents in 5 years (percentile)</th></tr>')
    rows = [f'<tr><td><a href="https://openalex.org/{esc(r.paper_id)}" target="_blank" rel="noopener">{esc(short_title(r, 120))}</a>'
            f'<div class="id">{esc(r.paper_id)}</div></td><td class="num">{esc(r.year_display)}</td><td class="num">{int(r.n_citing_inventions)}</td>'
            f'<td class="num">{int(r.n_citing_self)}</td><td class="num">{esc(first_cite.get(r.paper_id))}</td>{pbar(getattr(r, "pcs_C_5_pctl", None))}</tr>'
            for r in d.itertuples()]
    return f'<div class="tablewrap"><table>{head}{"".join(rows)}</table></div>'

tiles = [
    ('Works analysed', f'{len(R):,}', f'research works up to {YEAR_MAX}; {N_WORKS_ALL:,} in the whole record'),
    ('Patents', f'{len(UT):,}', f'granted up to {YEAR_MAX}; {N_PATENTS_ALL:,} in the whole record'),
    ('Median impact', f'{w_imp.median_pctl:.2f}' if w_imp.n else '–', 'cohort percentile of 5-year citations (0.50 = typical paper)'),
    ('Top 1 % works', f'{w_imp.share_top1:.0%}' if w_imp.n else '–', 'share of works in the top 1 % of their year × field'),
    ('Cited by books', f'{n_any:,}', f'works cited by a book or chapter; {n_text} by at least {BOOK_MIN}'),
    ('On both sides', f'{len(both_authors):,}', 'co-authors who are also co-inventors'),
    ('Cited by patents', f'{N_CITING:,}', f"distinct inventions citing the works; {N_CITING_SELF} of them the person's own"),
]
tiles_html = ''.join(f'<div class="tile"><div class="tlabel">{esc(a)}</div><div class="tvalue">{esc(b)}</div><div class="tnote">{esc(c)}</div></div>'
                     for a, b, c in tiles)
PERSON = f'{AUTHOR_NAME} ({AUTHOR_ID})'            # the person as shown on the page: name (OpenAlex author id)
no_pat = (f'No patents granted up to {YEAR_MAX}' + (f' ({N_PATENTS_ALL} granted later)' if N_PATENTS_ALL else '') + ': no co-invention network.')
TABS = [   # (key, label, [(figure, title, caption, note when the figure is missing)])
    ('overview', 'Overview', [('fig2_percentile_profile', 'Every document against its cohort (percentile profile)',
                               'One dot per work or patent: its percentile among the documents of the same year and field (works) or grant year and CPC section (patents). Bars show the interquartile range, ticks the median.', None),
                              ('fig1_output', 'Cumulative output', 'Works by publication year (blue: research works, grey: every document type) and utility patents by grant year, cumulated.', None)]),
    ('time', 'Over time', [('fig3_percentiles_by_year', 'Mean percentile per year', 'Impact, disruption, Foundation / Extension / Generalization and book citations; each panel is one measure, 0.5 is the cohort median.', None),
                           ('fig5_citation_trajectory', 'Citations received per year', 'Paper and patent citations by citing year. The OpenAlex reference graph is incomplete after 2022; the right panel removes that coverage effect.', None)]),
    ('books', 'Books', [('fig10_book_citations', 'Textbook reach: citations from books', f'Citations from books, book chapters and reference entries. {n_any} works are cited by at least one, {n_text} by {BOOK_MIN} or more.', None)]),
    ('patcites', 'Patent citations', [('figP_patent_citations', 'Patent → paper citations',
                                       f'US patents and pre-grant publications citing the works (Reliance on Science), counted as distinct inventions. {science_link}',
                                       'No patent cites the works.')]),
    ('networks', 'Networks', [('fig6_coauthorship_network', 'Co-authorship network', 'The person (star) and the co-authors on works with up to 50 authors, labelled name (OpenAlex author id); edges join co-authors who published together. Scroll to zoom, hover for details.', 'No co-authors on works with up to 50 authors.'),
                              ('fig7_coinvention_network', 'Co-invention network', 'The person (star) and the co-inventors on the patents, labelled name (PatentsView inventor id).', no_pat),
                              ('fig8_combined_network', 'Both networks combined', f'The union of the two networks above (top {HTML_TOP_K} of each side): co-authorship (blue edges) and co-invention (orange edges) in one graph; green people appear on both sides.', no_pat),
                              ('fig9_collaborator_evolution', 'New collaborators per year', 'By year of the first shared document.', None),
                              ('fig12_works_institution_network', 'Institution network (works)', f'Institutions of all authors on the works with up to {MAX_TEAM_SIZE} authors, labelled name (country); the person\'s own affiliations have a black rim. Size = works, edges join institutions on the same work.', 'No institutions recorded on the works.'),
                              ('fig14_patents_assignee_network', 'Assignee network (patents)', 'Owners of the patents (companies, universities, individuals); assignees matching the person\'s affiliations have a black rim. Size = patents, edges join co-assignees of the same patent.',
                               no_pat.replace('co-invention network', 'assignee network'))]),
]
nav = ''.join(f'<a href="#sec-{k}">{esc(lab)}</a>' for k, lab, _ in TABS) + '<a href="#sec-docs">Documents</a>'   # plain links: no script needed
panels, scripts = [], []
for k, lab, figs_ in TABS:
    cards = []
    for name, title, cap, note in figs_:
        j = fig_json(name)
        if j is None:
            if note:
                cards.append(f'<section class="card"><h3>{esc(title)}</h3><p class="muted">{esc(note)}</p></section>')
            continue
        cards.append(f'<section class="card"><h3>{esc(title)}</h3><p class="cap">{esc(cap)}</p><div class="plot" id="p-{name}" data-fig="{name}"></div></section>')
        scripts.append(f'<script type="application/json" id="f-{name}">{j}</script>')
    extra = ''
    if k == 'overview':
        extra = f'<section class="card"><h3>Percentile summary</h3><p class="cap">Median cohort percentile and the shares of documents in the cohort top 10 % and top 1 %, per measure.</p>{psum_table()}</section>'
    if k == 'books':
        extra = f'<section class="card"><h3>Where the citing books come from</h3>{books_table()}</section>'
    post = ''
    if k == 'patcites':
        post = f'<section class="card"><h3>Works most cited by patents</h3><p class="cap">Distinct citing inventions per work, the person\'s own among them, and the year of the first citing patent.</p>{patent_cited_table()}</section>'
    panels.append(f'<section class="section" id="sec-{k}"><h2>{esc(lab)}</h2>{extra}{"".join(cards) or "<p class=muted>No data for this view.</p>"}{post}</section>')
panels.append(f'<section class="section" id="sec-docs"><h2>Documents</h2><section class="card"><h3>Works with the highest impact percentile</h3>{works_table()}</section>'
              f'<section class="card"><h3>Patents with the highest impact percentile</h3>{patents_table()}</section></section>')

PAGE = f'''<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">
<title>{esc(PERSON)} Research Profile</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;500&family=IBM+Plex+Sans:wght@400;500;600&family=Source+Serif+4:opsz,wght@8..60,600&display=swap">
<style>
:root {{ --ground:#f4f6f5; --surface:#ffffff; --ink:#16201c; --muted:#5a6761; --rule:#dce3df; --soft:#eef2f0;
        --works:#2a78d6; --patents:#eb6834; --links:#1baf7a; color-scheme: light; }}
@media (prefers-color-scheme: dark) {{ :root:not([data-theme="light"]) {{ --ground:#111614; --surface:#18201c; --ink:#e6ebe8; --muted:#9daea6;
        --rule:#2b3631; --soft:#1f2924; --works:#3987e5; --patents:#d95926; --links:#199e70; color-scheme: dark; }} }}
:root[data-theme="dark"] {{ --ground:#111614; --surface:#18201c; --ink:#e6ebe8; --muted:#9daea6; --rule:#2b3631; --soft:#1f2924;
        --works:#3987e5; --patents:#d95926; --links:#199e70; color-scheme: dark; }}
html, body {{ margin: 0; }}
body {{ background: var(--ground); color: var(--ink); font: 15px/1.5 'IBM Plex Sans', system-ui, -apple-system, 'Segoe UI', sans-serif; }}
.wrap {{ max-width: 1280px; margin: 0 auto; padding-inline: 20px; padding-block: 28px 48px; display: grid; gap: 22px; }}
header {{ display: grid; gap: 6px; }}
.eyebrow {{ font: 500 12px 'IBM Plex Mono', ui-monospace, monospace; letter-spacing: .08em; text-transform: uppercase; color: var(--muted); }}
h1 {{ font: 600 clamp(28px, 4vw, 40px)/1.1 'Source Serif 4', Georgia, serif; margin: 0; text-wrap: balance; }}
.ids {{ font: 13px 'IBM Plex Mono', ui-monospace, monospace; color: var(--muted); display: flex; flex-wrap: wrap; gap: 6px 16px; }}
.ids a {{ color: inherit; }}
.scope {{ color: var(--muted); max-width: 75ch; }}
.tiles {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(180px, 1fr)); gap: 12px; }}
.tile {{ background: var(--surface); border: 1px solid var(--rule); border-radius: 10px; padding: 14px 16px; display: grid; gap: 2px; align-content: start; }}
.tlabel {{ font-size: 12px; letter-spacing: .04em; text-transform: uppercase; color: var(--muted); }}
.tvalue {{ font: 600 28px/1.15 'IBM Plex Sans', sans-serif; font-variant-numeric: tabular-nums; }}
.tnote {{ font-size: 12.5px; color: var(--muted); }}
html {{ scroll-behavior: smooth; }}
nav.toc {{ position: sticky; top: 0; z-index: 10; display: flex; gap: 2px; overflow-x: auto; background: var(--ground);
          border-bottom: 1px solid var(--rule); padding-block: 6px; }}
nav.toc a {{ font: 500 14px 'IBM Plex Sans', sans-serif; color: var(--muted); text-decoration: none; padding: 8px 12px; border-radius: 6px; white-space: nowrap; }}
nav.toc a:hover, nav.toc a:focus-visible {{ color: var(--ink); background: var(--soft); outline: none; }}
.section {{ display: grid; gap: 18px; scroll-margin-top: 64px; }}
.section > h2 {{ margin: 18px 0 0; font: 600 22px 'Source Serif 4', Georgia, serif; }}
.card {{ background: var(--surface); border: 1px solid var(--rule); border-radius: 10px; padding: 16px 18px; min-width: 0; }}
.card h3 {{ margin: 0 0 2px; font: 600 17px 'IBM Plex Sans', sans-serif; }}
.cap {{ margin: 0 0 10px; color: var(--muted); font-size: 13.5px; max-width: 95ch; }}
.plot {{ width: 100%; min-height: 360px; }}
.tablewrap {{ overflow-x: auto; }}
table {{ border-collapse: collapse; width: 100%; font-size: 13.5px; }}
th, td {{ text-align: left; padding: 7px 10px; border-bottom: 1px solid var(--rule); vertical-align: top; }}
th {{ font-weight: 500; color: var(--muted); font-size: 12px; letter-spacing: .03em; text-transform: uppercase; }}
.num {{ text-align: right; font-variant-numeric: tabular-nums; white-space: nowrap; }}
.muted {{ color: var(--muted); }}
.id {{ font: 11.5px 'IBM Plex Mono', monospace; color: var(--muted); }}
td a {{ color: var(--ink); text-decoration-color: var(--rule); }}
.pbar {{ display: inline-block; width: 56px; height: 6px; margin-right: 8px; vertical-align: middle; background: var(--soft); border-radius: 3px; overflow: hidden; }}
.pbar i {{ display: block; height: 100%; background: var(--works); }}
footer {{ color: var(--muted); font-size: 12.5px; max-width: 95ch; }}
@media (prefers-reduced-motion: reduce) {{ * {{ transition: none !important; }} html {{ scroll-behavior: auto; }} }}
</style>
</head>
<body>
<div class="wrap">
<header>
  <div class="eyebrow">{esc(nobel_line)}Research profile · Science of Science metrics · up to {YEAR_MAX}</div>
  <h1>{esc(PERSON)}</h1>
  <div class="ids"><span>OpenAlex {esc(', '.join(AUTHOR_IDS))}</span>{''.join(f'<a href="{esc(o)}" target="_blank" rel="noopener">ORCID {esc(o.rsplit("/", 1)[-1])}</a>' for o in orcids)}
    <span>PatentsView {esc(', '.join(inv_sel) or '—')}{esc(f' ({INVENTOR_SOURCE})' if inv_sel else '')}</span></div>
  <p class="scope">{esc(top_aff)}. Every measure is a percentile among documents of the same cohort: works against papers of the same publication
  year and field, patents against patents of the same grant year and CPC section. Works published and patents granted up to {YEAR_MAX}.</p>
</header>
<div class="tiles">{tiles_html}</div>
<nav class="toc" aria-label="Sections">{nav}</nav>
{''.join(panels)}
<footer>Sources: OpenAlex (2026-01 snapshot), PatentsView (2025-12-31), Reliance on Science (patent-to-paper citations), SciSciNet, and the
Science of Science pipelines (citations, disruption CD with Foundation / Extension / Generalization, citations from books). Built by notebook/author_profile.ipynb on {datetime.date.today().isoformat()}.</footer>
</div>
{''.join(scripts)}
{PLOTLY_TAG}
<script>
(function () {{
  const css = n => getComputedStyle(document.documentElement).getPropertyValue(n).trim();
  const drawn = new Set();
  function themed(layout) {{
    const L = Object.assign({{}}, layout);
    L.paper_bgcolor = 'rgba(0,0,0,0)'; L.plot_bgcolor = 'rgba(0,0,0,0)';
    L.font = Object.assign({{}}, L.font || {{}}, {{ color: css('--ink'), family: "'IBM Plex Sans', system-ui, sans-serif" }});
    L.hoverlabel = {{ bgcolor: css('--surface'), bordercolor: css('--rule'), font: {{ color: css('--ink') }} }};
    L.legend = Object.assign({{}}, L.legend || {{}}, {{ bgcolor: 'rgba(0,0,0,0)' }});
    Object.keys(L).forEach(k => {{ if (/^[xy]axis\\d*$/.test(k)) L[k] = Object.assign({{}}, L[k], {{ gridcolor: css('--rule'), zerolinecolor: css('--rule'), linecolor: css('--rule') }}); }});
    (L.annotations || []).forEach(a => {{ a.font = Object.assign({{}}, a.font || {{}}, {{ color: css('--ink') }}); }});
    L.autosize = true; delete L.width;
    return L;
  }}
  function draw(el) {{
    if (drawn.has(el.id) || !window.Plotly) return;
    const src = document.getElementById('f-' + el.dataset.fig);
    if (!src) return;
    try {{
      const fig = JSON.parse(src.textContent);
      el.style.minHeight = ((fig.layout && fig.layout.height) || 420) + 'px';
      Plotly.newPlot(el, fig.data, themed(fig.layout || {{}}), {{ responsive: true, displaylogo: false, scrollZoom: false }});
      drawn.add(el.id);
    }} catch (e) {{ el.textContent = 'This figure could not be drawn: ' + e.message; }}
  }}
  function retheme() {{ drawn.forEach(id => {{ const el = document.getElementById(id); const f = JSON.parse(document.getElementById('f-' + el.dataset.fig).textContent);
                                             Plotly.relayout(el, themed(f.layout || {{}})); }}); }}
  const plots = Array.from(document.querySelectorAll('.plot'));
  if (!window.Plotly) plots.forEach(el => {{ el.textContent = 'plotly.js did not load.'; }});
  if (window.Plotly) plots.forEach(draw);          // every figure is drawn at load, synchronously: nothing waits for a click or a timer
  window.matchMedia('(prefers-color-scheme: dark)').addEventListener('change', retheme);
  new MutationObserver(retheme).observe(document.documentElement, {{ attributes: true, attributeFilter: ['data-theme'] }});
}})();
</script>
</body>
</html>
'''
# output/dashboard/<prize year>_<field>_<name>_<author id>.html, a standalone page (NA where the author is not a laureate)
DASH_PATH = npc.dashboard_path(NOBEL.get('year'), NOBEL.get('field'), NOBEL.get('name') or AUTHOR_NAME, AUTHOR_ID)
tmp = DASH_PATH.with_name(DASH_PATH.name + '.tmp')
tmp.write_text(PAGE, encoding='utf-8'); os.replace(tmp, DASH_PATH); WRITTEN.append(DASH_PATH)
if (OUT / 'dashboard.html').exists():             # the location used before 2026-09-30
    (OUT / 'dashboard.html').unlink(); print('removed the earlier', (OUT / 'dashboard.html').relative_to(npc.NP))
print(f'dashboard -> {DASH_PATH.relative_to(npc.NP)}  ({DASH_PATH.stat().st_size / 1e6:.1f} MB)')

## 10b. Record for agents

The dashboard's content as Markdown, `Nobel Prize/output/record/<same name as the dashboard>.md`: a YAML header (person, ids,
Nobel prizes, window, sources, dashboard path; values are JSON, which is valid YAML) and sections with the numbers and tables
behind every block of the dashboard, plus the definitions an agent needs to read them.

In [ ]:
# output/record/<same name as the dashboard>.md: the dashboard's content as Markdown with a YAML header, for agents
def md_esc(x):
    s = '' if x is None or x is pd.NA or (isinstance(x, float) and not math.isfinite(x)) else str(x)
    return s.replace('|', '\\|').replace('\n', ' ').strip()

def md_num(x, kind):
    if x is None or x is pd.NA or (isinstance(x, float) and not math.isfinite(x)) or pd.isna(x):
        return ''
    if kind == 'p':
        return f'{float(x):.3f}'
    if kind == '%':
        return f'{float(x):.1%}'
    if kind == 'n':
        return f'{int(round(float(x))):,}'
    if kind == 'y':
        return str(int(float(x)))
    return md_esc(x)

def md_table(df, cols, n=None):
    # cols: [(column, header, kind)]; kind 'p' percentile (3 decimals), '%' share, 'n' count, 'y' year, 't' text
    if df is None or not len(df):
        return '_none_\n'
    d = df.head(n) if n else df
    head = '| ' + ' | '.join(h for _, h, _ in cols) + ' |\n| ' + ' | '.join('---' if k == 't' else '---:' for _, _, k in cols) + ' |\n'
    body = ''.join('| ' + ' | '.join(md_esc(r.get(c)) if k == 't' else md_num(r.get(c), k) for c, _, k in cols) + ' |\n'
                   for _, r in d.iterrows())
    return head + body

def title_of(pid, k=200):
    t = papers.set_index('paper_id').title.get(pid) if 'title' in papers else None
    return (t[:k] if isinstance(t, str) and t else '')

RECORD_DIR = npc.RECORD_DIR
RECORD_DIR.mkdir(parents=True, exist_ok=True)
REC_PATH = RECORD_DIR / (DASH_PATH.stem + '.md')
lab_w = {m[0]: m[1] for m in PAPER_MEASURES}
front = {
    'record_type': 'research_profile',
    'schema_version': 1,
    'query': ({'name': QUERY, 'resolution': RESOLUTION} if QUERY else None),
    'person': {'name': AUTHOR_NAME, 'display': PERSON, 'openalex_author_ids': AUTHOR_IDS, 'input_author_ids': INPUT_AUTHOR_IDS,
               'author_id_check': ID_CHECK_RESULT.get('rule'), 'orcid': [o.rstrip('/').rsplit('/', 1)[-1] for o in orcids],
               'patentsview_inventor_ids': inv_sel, 'inventor_source': INVENTOR_SOURCE},
    'nobel': ({k: NOBEL.get(k) for k in ('prizes', 'year', 'field', 'name', 'wikidata_qid', 'prizeatlas_url', 'rule')} if NOBEL else None),
    'window': {'works_published_up_to': YEAR_MAX, 'patents_granted_up_to': YEAR_MAX, 'window_measures_closed_by': WINDOW_END_YEAR},
    'dashboard': str(DASH_PATH.relative_to(npc.NP)),
    'outputs_folder': str(OUT.relative_to(npc.NP)),
    'generated': datetime.date.today().isoformat(),
    'generator': 'Nobel Prize/notebook/author_profile.ipynb',
    'sources': ['OpenAlex 2026-01 snapshot (works, authorships, citations, book citations)', 'PatentsView 2025-12-31 (patents, inventors, assignees)',
                'Reliance on Science (patent-to-paper citations)', 'pqrs author-inventor crosswalk / PatentsView name search',
                'Science of Science pipelines (citation windows, disruption CD, Foundation / Extension / Generalization)', 'PrizeAtlas (Nobel prizes)'],
}
L_ = ['---'] + [f'{k}: {json.dumps(v, ensure_ascii=False, default=str)}' for k, v in front.items()] + ['---', '']   # JSON values are valid YAML
L_ += [f'# {PERSON}: research profile', '']
if NOBEL.get('prizes'):
    L_ += [f"Nobel Prize: {', '.join(NOBEL['prizes'])}. Affiliations with most works: {top_aff}.", '']

# at a glance
w_imp5 = PSUM[(PSUM.side == 'works') & (PSUM.key == 'impact5')]
ca_small = coauth_nodes[~coauth_nodes.only_large_teams] if len(coauth_nodes) else coauth_nodes
ci_all = coinv_nodes if len(coinv_nodes) else coinv_nodes
glance = [
    f'- Works analysed: {len(R):,} research works (article, review, letter) published up to {YEAR_MAX}; {N_WORKS_ALL:,} works in the whole OpenAlex record.',
    f'- Patents: {len(UT):,} US utility patents granted up to {YEAR_MAX}; {N_PATENTS_ALL:,} patents in the whole record; inventor ids {", ".join(inv_sel) or "none"} ({INVENTOR_SOURCE}).',
    (f'- Impact: median 5-year citation percentile {w_imp5.median_pctl.iloc[0]:.3f}; {w_imp5.share_top10.iloc[0]:.1%} of the works in the cohort top 10 %, '
     f'{w_imp5.share_top1.iloc[0]:.1%} in the top 1 %.') if len(w_imp5) and w_imp5.n.iloc[0] else '- Impact: no work with a 5-year citation percentile.',
    f'- Textbook reach: {n_any:,} works cited by books, book chapters or reference entries; {n_text:,} by at least {BOOK_MIN}; '
    f'{books.book_id.nunique() if len(books) else 0:,} distinct citing books.',
    f'- Patent → paper citations: {N_CITING:,} distinct inventions cite the works, {N_CITING_SELF:,} of them the person\'s own. {science_link}'.rstrip(),
    f'- Collaboration: {len(ca_small):,} co-authors on works with up to {MAX_TEAM_SIZE} authors, {len(ci_all):,} co-inventors, '
    f'{len(both_authors):,} people on both sides.',
]
L_ += ['## At a glance', ''] + glance + ['']

# percentile summary (the dashboard's measures)
ps = []
for k in DASH_MEASURES:
    w = PSUM[(PSUM.side == 'works') & (PSUM.key == k)]; t = PSUM[(PSUM.side == 'patents') & (PSUM.key == k)]
    row = {'measure': (w.measure.iloc[0] if len(w) else (t.measure.iloc[0] if len(t) else k))}
    for side, part in (('works', w), ('patents', t)):
        if len(part) and part.n.iloc[0] > 0:
            r = part.iloc[0]
            row.update({f'{side}_n': r.n, f'{side}_median': r.median_pctl, f'{side}_top10': r.share_top10, f'{side}_top1': r.share_top1})
    ps.append(row)
L_ += ['## Percentile summary', '',
       'Median cohort percentile and shares of documents in the cohort top 10 % / top 1 % (0.5 = the typical document of the same cohort).', '',
       md_table(pd.DataFrame(ps), [('measure', 'Measure', 't'), ('works_median', 'Works: median', 'p'), ('works_top10', 'top 10 %', '%'),
                                   ('works_top1', 'top 1 %', '%'), ('works_n', 'n', 'n'), ('patents_median', 'Patents: median', 'p'),
                                   ('patents_top10', 'top 10 %', '%'), ('patents_top1', 'top 1 %', '%'), ('patents_n', 'n', 'n')])]

# output and percentiles per year
out_y = pd.DataFrame({'research_works': cnt_p.research, 'all_works': cnt_p.research + cnt_p.other}) if len(cnt_p) else pd.DataFrame()
if len(fy):
    out_y = out_y.join(fy.rename('patents'), how='outer')
out_y = out_y.fillna(0).sort_index()
if len(out_y):
    out_y = out_y.assign(**{f'cum_{c}': out_y[c].cumsum() for c in out_y.columns}).reset_index().rename(columns={'index': 'year'})
    out_y = out_y.rename(columns={out_y.columns[0]: 'year'})
cols_o = [('year', 'Year', 'y'), ('research_works', 'Research works', 'n'), ('all_works', 'All works', 'n')] + \
         ([('patents', 'Patents', 'n')] if 'patents' in out_y else []) + [('cum_research_works', 'Cumulative research works', 'n')] + \
         ([('cum_patents', 'Cumulative patents', 'n')] if 'cum_patents' in out_y else [])
L_ += ['## Output per year', '', 'Works by publication year, utility patents by grant year.', '', md_table(out_y, cols_o)]
yw = yearly[[c for c in [f'works_{m[2]}' for m in PAPER_MEASURES if m[0] in DASH_MEASURES] + ['n_works'] if c in yearly]].dropna(how='all') if 'yearly' in globals() else pd.DataFrame()
if len(yw):
    yw = yw.reset_index().rename(columns={yw.index.name or 'index': 'year'})
    yw = yw.rename(columns={yw.columns[0]: 'year'})
    L_ += ['## Mean percentile per year (works)', '', 'Mean cohort percentile of the year\'s works; 0.5 = the cohort median.', '',
           md_table(yw, [('year', 'Year', 'y'), ('n_works', 'Works', 'n')] + [(f'works_{m[2]}', m[1], 'p') for m in PAPER_MEASURES
                                                                              if m[0] in DASH_MEASURES and f'works_{m[2]}' in yw])]
if 'traj' in globals() and len(traj):
    tj = traj.reset_index()
    tj = tj[pd.to_numeric(tj.cite_year, errors='coerce') <= YEAR_MAX]
    L_ += ['## Citations received per year', '', f'By citing year up to {YEAR_MAX}.', '',
           md_table(tj, [('cite_year', 'Year', 'y'), ('p2p', 'Paper → paper citations to the works', 'n'), ('pcs_all_works', 'Patent → paper citations to the works', 'n'),
                         ('uniqueC', 'Citations to the patents (uniqueC)', 'n')])]

# top works and patents
tw = R.sort_values(['C_5_pctl', 'C_all'], ascending=False).head(25).assign(title_=lambda x: x.paper_id.map(title_of))
L_ += ['## Works with the highest impact percentile', '',
       md_table(tw, [('title_', 'Title', 't'), ('paper_id', 'Work id', 't'), ('year', 'Year', 'y'), ('journal', 'Venue', 't'), ('C_5_pctl', 'Impact', 'p'),
                     ('CD_5_rpctl_mid', 'Disruption', 'p'), ('F_5_pctl', 'Foundation', 'p'), ('E_5_pctl', 'Extension', 'p'), ('G_5_pctl', 'Generalization', 'p'),
                     ('book_cites_pctl', 'Books', 'p'), ('n_book_cites', 'Citing books', 'n'), ('pcs_C_total', 'Citing patents', 'n')])]
tp = UT.sort_values('uniqueC_5_pctl', ascending=False).head(20) if len(UT) else UT
L_ += ['## Patents with the highest impact percentile', '',
       md_table(tp, [('patent_title', 'Title', 't'), ('patent_id', 'Patent', 't'), ('grant_year', 'Grant year', 'y'), ('uniqueC_5_pctl', 'Impact', 'p'),
                     ('CD_5_rpctl_mid', 'Disruption', 'p'), ('F_5_pctl', 'Foundation', 'p'), ('E_5_pctl', 'Extension', 'p'), ('G_5_pctl', 'Generalization', 'p'),
                     ('n_paper_refs', 'Papers cited', 'n')]) if len(tp) else f'_No US utility patents granted up to {YEAR_MAX}._\n']

# books
if len(books):
    bs = books.drop_duplicates('book_id').book_source.replace('', np.nan).dropna().value_counts().head(12).rename_axis('source').reset_index(name='books')
    bw_ = papers[papers.n_book_cites > 0].sort_values(['n_book_cites', 'C_all'], ascending=False).head(15).assign(title_=lambda x: x.paper_id.map(title_of))
    by = books.drop_duplicates(['book_id', 'paper_id']).groupby('book_year').size().rename('book_citations').reset_index()
    by = by[pd.to_numeric(by.book_year, errors='coerce') <= YEAR_MAX]
    L_ += ['## Textbook reach: citations from books', '',
           f'Citing works of OpenAlex type book, book-chapter, book-section or reference-entry. {n_any:,} works are cited by at least one, {n_text:,} by {BOOK_MIN} or more.', '',
           '### Where the citing books come from', '', md_table(bs, [('source', 'Source (series / publisher)', 't'), ('books', 'Books', 'n')]),
           '### Works most cited by books', '', md_table(bw_, [('title_', 'Title', 't'), ('paper_id', 'Work id', 't'), ('year_display', 'Year', 'y'),
                                                               ('n_book_cites', 'Citing books / chapters / entries', 'n'), ('book_cites_pctl', 'Book-citation percentile', 'p')]),
           '### Book citations per year', '', md_table(by, [('book_year', 'Year', 'y'), ('book_citations', 'Book citations', 'n')])]
else:
    L_ += ['## Textbook reach: citations from books', '', '_No citing books._', '']

# patent -> paper citations
if N_CITING:
    pw_ = papers[papers.n_citing_inventions > 0].sort_values(['n_citing_inventions', 'C_all'], ascending=False).head(20).assign(
        title_=lambda x: x.paper_id.map(title_of), first_patent=lambda x: x.paper_id.map(first_cite))
    iy = (cites_in.dropna(subset=['cite_year']).groupby('invention_id').agg(cite_year=('cite_year', 'min'), own=('self_citation', 'max'))
          .groupby('cite_year').agg(inventions=('own', 'size'), own=('own', 'sum')).reset_index())
    iy = iy[iy.cite_year <= YEAR_MAX]
    L_ += ['## Patent → paper citations', '', f'US patents and pre-grant publications citing the works (Reliance on Science), as distinct inventions. {science_link}', '',
           '### Works most cited by patents', '',
           md_table(pw_, [('title_', 'Title', 't'), ('paper_id', 'Work id', 't'), ('year_display', 'Year', 'y'), ('n_citing_inventions', 'Citing inventions', 'n'),
                          ('n_citing_self', 'Own patents', 'n'), ('first_patent', 'First patent citation', 'y'), ('pcs_C_5_pctl', 'Cited by patents in 5 years (percentile)', 'p')]),
           '### Citing inventions per year', '', md_table(iy, [('cite_year', 'Year', 'y'), ('inventions', 'Citing inventions', 'n'), ('own', 'Own patents', 'n')])]
else:
    L_ += ['## Patent → paper citations', '', '_No patent cites the works._', '']

# collaboration
L_ += ['## Collaboration networks', '', f'The person is the centre (ego) of each network; collaborators on works with more than {MAX_TEAM_SIZE} authors are left out.', '',
       '### Co-authors (top 30)', '', md_table(ca_small, [('name', 'Name', 't'), ('id', 'OpenAlex author id', 't'), ('n_shared', 'Shared works', 'n'),
                                                       ('first_year', 'First', 'y'), ('last_year', 'Last', 'y'), ('also_coinventor', 'Also co-inventor', 't')], n=30)]
L_ += ['### Co-inventors (top 30)', '', md_table(ci_all, [('name', 'Name', 't'), ('id', 'PatentsView inventor id', 't'), ('n_shared', 'Shared patents', 'n'),
                                                        ('first_year', 'First', 'y'), ('last_year', 'Last', 'y'), ('also_coauthor', 'Also co-author', 't')], n=30)
       if len(ci_all) else f'_No co-inventors (no patents granted up to {YEAR_MAX})._\n']
L_ += ['### People on both sides (co-author and co-inventor)', '',
       md_table(ok_ov, [('author_name', 'Name', 't'), ('author_id', 'OpenAlex author id', 't'), ('inventor_id', 'PatentsView inventor id', 't'),
                        ('n_shared_works', 'Shared works', 'n'), ('n_shared_patents', 'Shared patents', 'n'), ('match_source', 'Matched by', 't')])
       if len(ok_ov) else '_none_\n']
newc = ca_small.dropna(subset=['first_year']).groupby('first_year').size().rename('new_coauthors') if len(ca_small) else pd.Series(dtype=int)
newi = ci_all.dropna(subset=['first_year']).groupby('first_year').size().rename('new_coinventors') if len(ci_all) else pd.Series(dtype=int)
nc = pd.concat([newc, newi], axis=1).fillna(0).sort_index().reset_index().rename(columns={'index': 'year', 'first_year': 'year'}) if len(newc) or len(newi) else pd.DataFrame()
if len(nc):
    nc = nc.rename(columns={nc.columns[0]: 'year'})
    L_ += ['### New collaborators per year', '', md_table(nc, [('year', 'Year', 'y')] + [(c, c.replace('_', ' ').capitalize(), 'n') for c in ('new_coauthors', 'new_coinventors') if c in nc])]
for key, head, idh in [('works_institution', 'Institutions on the works (top 30)', 'OpenAlex institution id'), ('patents_assignee', 'Assignees of the patents (top 20)', 'PatentsView assignee id')]:
    res_ = ENTITY.get(key)
    if res_ is None or not len(res_[0]):
        L_ += [f'### {head}', '', '_none_', '']; continue
    nd = res_[0].assign(own=lambda x: np.where(x.id.isin(OWN.get(key, set())), 'yes', ''))
    L_ += [f'### {head}', '', md_table(nd, [('name', 'Name', 't'), ('id', idh, 't'), ('n_shared', 'Shared documents', 'n'), ('first_year', 'First', 'y'),
                                              ('last_year', 'Last', 'y'), ('own', "The person's affiliation", 't')], n=30 if key == 'works_institution' else 20)]
if len(stats):
    L_ += ['### Network statistics', '', md_table(stats.reset_index(drop=True), [(c, c, 't' if c == 'layer' else 'p') for c in stats.columns])]

# definitions
L_ += ['## Definitions and caveats', '',
       '- Percentile: the mid-rank share of the document\'s cohort below it (ties split evenly), 0 = lowest, 0.5 = typical, 1 = highest. Works are compared with '
       'the OpenAlex papers of the same publication year and first field (FoS); patents with the US utility patents of the same grant year and CPC section.',
       '- Top 10 % / top 1 %: percentile of at least 0.90 / 0.99.',
       '- Impact: citations received in the first 5 (or 10) years after publication; for patents, citations from granted patents and pre-grant publications (uniqueC).',
       '- Disruption (CD, 5 years): how far the document eclipses its references among the papers that cite it within 5 years; only documents with references.',
       '- Foundation / Extension / Generalization (5 years): shares of the citing documents that build on the document itself (foundation), extend it together '
       'with its references (extension), or generalize over them.',
       f'- Textbook reach: citations from OpenAlex books, book chapters, book sections and reference entries; "cited by at least {BOOK_MIN} books".',
       '- Patent → paper citations: Reliance on Science links from US patents and pre-grant publications to OpenAlex works, counted as distinct inventions; '
       '"own patents" are the person\'s patents.',
       f'- Window: works published and patents granted up to {YEAR_MAX}; yearly series end in {YEAR_MAX}. About a quarter of the works in an OpenAlex author record '
       'are missing from the 2026-01 works table and carry no metrics (their titles come from the OpenAlex API).',
       f'- Author id check: {ID_CHECK_RESULT.get("rule")}; inventor ids: {INVENTOR_SOURCE}.', '']
tmp = REC_PATH.with_name(REC_PATH.name + '.tmp')
tmp.write_text('\n'.join(L_), encoding='utf-8'); os.replace(tmp, REC_PATH); WRITTEN.append(REC_PATH)
print(f'record -> {REC_PATH.relative_to(npc.NP)}  ({REC_PATH.stat().st_size / 1e3:.0f} kB)')

## 11. Manifest

Parameters, the fingerprint of every input, which per-author caches were reused or rebuilt, timings
and the files this run wrote.

In [ ]:
SOURCES = {p.name: p for p in PAPER_SOURCES + PATENT_SOURCES + AUTH_SOURCES} | {
    'pqrs_dataset.tsv': npc.PQRS_TSV, 'paper_da_cache': npc.PAPER_DA_CACHE, 'patent_master': npc.PATENT_MASTER, 'pcs_oa_uspto.csv': npc.PCS_CSV,
    'paper_citation_trend': npc.OA / 'paper_citation_trend.parquet', 'patent_citation_trend': npc.PV / 'patent_citation_trend.parquet',
    'patent_uniqueC_trend': npc.PV / 'patent_uniqueC_trend.parquet', 'g_inventor_disambiguated': npc.granted('g_inventor_disambiguated.tsv.zip'),
    'g_patent': npc.granted('g_patent.tsv.zip'), 'g_persistent_inventor': npc.granted('g_persistent_inventor.tsv.zip'),
    'pg_granted_pgpubs_crosswalk': npc.pregranted('pg_granted_pgpubs_crosswalk.tsv.zip'), 'pg_inventor_disambiguated': npc.pregranted('pg_inventor_disambiguated.tsv.zip'),
    'institutions': npc.OA_RAW / 'institutions.csv.gz'}
SHARED = sorted(p for p in npc.CACHE.glob('*.parquet'))
SOURCES |= {'paper_z_score': npc.OA / 'paper_z_score.parquet', 'patent_z_score': npc.PV / 'patent_z_score.parquet'}
manifest = {
    'finished_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'slurm_job_id': os.environ.get('SLURM_JOB_ID'),
    'parameters': dict(AUTHOR_ID=AUTHOR_ID, EXTRA_AUTHOR_IDS=EXTRA_AUTHOR_IDS, YEAR_MAX=YEAR_MAX, WINDOW_END_YEAR=WINDOW_END_YEAR, BOOK_MIN=BOOK_MIN,
                       PATENT_YEAR_BASIS=PATENT_YEAR_BASIS,
                       HTML_TOP_K=HTML_TOP_K, INVENTOR_IDS=INVENTOR_IDS, INVENTOR_SELECT=INVENTOR_SELECT,
                       MIN_CONFIDENCE=MIN_CONFIDENCE, MIN_RELEASE_SHARE=MIN_RELEASE_SHARE, NAME_SEARCH=NAME_SEARCH, RUN_TRAJECTORY=RUN_TRAJECTORY,
                       FETCH_TITLES=FETCH_TITLES,
                       TOP_K_COLLAB=TOP_K_COLLAB, MAX_TEAM_SIZE=MAX_TEAM_SIZE, OBS_END_YEAR=OBS_END_YEAR, COVERAGE_END=COVERAGE_END,
                       CD_REQUIRE_REFS=CD_REQUIRE_REFS, RESEARCH_ONLY=RESEARCH_ONLY, REBUILD_CACHE=REBUILD_CACHE, CACHE_VERSION=CACHE_VERSION,
                       PRIZE_YEAR=PRIZE_YEAR, PRIZE_FIELD=PRIZE_FIELD, ID_CHECK=ID_CHECK, MIN_WORKS=MIN_WORKS, TITLES_TABLE=TITLES_TABLE,
                       NOBEL_LOOKUP=NOBEL_LOOKUP, QUERY=QUERY, RESOLUTION=RESOLUTION,
                       THREADS=THREADS, DUCKDB_MEM=DUCKDB_MEM),
    'summary': summary, 'network_stats': stats.to_dict('records'), 'timings_s': TIMINGS,
    'dashboard': str(DASH_PATH.relative_to(npc.NP)), 'nobel_identity': NOBEL, 'record': str(REC_PATH.relative_to(npc.NP)),
    'author_id_check': ID_CHECK_RESULT,
    'per_author_caches': cache.log,
    'sources': {k: npc.fingerprint(v) for k, v in SOURCES.items()},
    'shared_caches': {p.name: npc.fingerprint(p) for p in SHARED},
}
if (HTML / 'plotly.min.js').exists():
    WRITTEN.append(HTML / 'plotly.min.js')
manifest['outputs'] = [{'file': os.path.relpath(p, OUT), 'bytes': Path(p).stat().st_size} for p in sorted(set(map(str, WRITTEN)))]
npc.write_json(OUT / 'manifest.json', manifest)
stale = sorted(str(p.relative_to(OUT)) for p in OUT.rglob('*') if p.is_file() and '_cache' not in p.parts
               and str(p) not in set(map(str, WRITTEN)) and p.name not in ('manifest.json',) and not p.name.endswith('.ipynb'))
print(json.dumps({'finished_utc': manifest['finished_utc'], 'timings_s': TIMINGS, 'caches': cache.log}, indent=1, default=str))
if stale:
    print('files in the output folder not written by this run (left from an earlier run):', stale)